# Environment

***Environment Setup Explanation***

This section establishes the foundation for our agent-based model by:

1. **Importing Libraries**: We're using a variety of Python libraries to handle different aspects of the model:
   - **GIS Tools**: ArcPy, GeoPandas, and Shapely for geospatial operations
   - **Data Analysis**: Pandas and NumPy for data manipulation
   - **Visualization**: Matplotlib and Seaborn for creating plots and maps
   - **Agent-Based Modeling**: Mesa framework for creating and simulating agents
   - **Scientific Computing**: SciPy for mathematical operations and Optuna for optimization

2. **Setting Up the Study Area**:
   - We define the geographic boundaries (extent) of our study area
   - This ensures all spatial operations are constrained to our region of interest

3. **Loading Input Data**:
   - The model requires multiple environmental datasets like elevation, slope, rainfall, etc.
   - These datasets represent the physical landscape where our agents will operate

4. **Defining Output Locations**:
   - We set up directories where simulation results will be stored

5. **Setting Cell Size**:
   - We're resampling our data to a coarser resolution (250m cells instead of the original)
   - This improves computational efficiency while maintaining sufficient detail
   - The code calculates how many rows and columns our simulation grid will have



In [ ]:
#Setup
import arcpy                       # For GIS processing and spatial analysis
from datetime import datetime      # For date and time operations
from scipy.signal import convolve2d
from arcpy.sa.ComplexArguments import FuzzyMSSmall
from sympy import shape
from tqdm import tqdm
import math
import time
import folium                      # For interactive map creation
import pyproj                      # For coordinate system transformations
import json                        # For JSON data handling
import optuna                      # For hyperparameter optimization
import numpy as np                 # For numerical operations
import os                          # For operating system interactions
import pickle                      # For serializing Python objects
from arcpy.sa import *             # Additional spatial analysis tools
import pandas as pd                # For data manipulation and analysis
import geopandas as gpd            # For geospatial data operations
import fiona                       # For reading/writing spatial data
import mesa                        # For agent-based modeling
from mesa.datacollection import DataCollector  # For collecting model data
import random                      # For random number generation
import copy                        # For creating deep copies of objects
import seaborn as sns              # For statistical data visualization
import matplotlib.pyplot as plt    # For plotting
import matplotlib.animation as animation  # For creating animations
from matplotlib.patches import Ellipse  # For drawing ellipses
from matplotlib import patches     # For drawing shapes
#import mesa_geo as mg             # Commented out GIS extension for Mesa
import pysal                       # For spatial analysis
from mesa.visualization import SolaraViz, make_plot_component, make_space_component  # For visualization
from sklearn.cluster import DBSCAN  # For density-based clustering
from scipy import linalg           # For linear algebra operations
from shapely.geometry import Point, Polygon  # For geometric operations
from scipy.optimize import minimize  # For optimization
from scipy.ndimage import distance_transform_edt  # For distance calculations
import optuna # For optimization



# Allow overwriting of existing files
arcpy.env.overwriteOutput = True
# Set the workspace (geodatabase) for ArcPy operations
arcpy.env.workspace =  r"C:\Users\Owner\Documents\Itay\Prj_25.gdb" #change to relevant
# Read shapefile containing geographic boundaries
gdf = gpd.read_file(r"C:\Users\Owner\Documents\Itay\thesis\HN_ext.shp")

# Extract the coordinates from the geometry to define the study area bounds
coords = []
for geometry in gdf.geometry:
    x, y = geometry.exterior.coords.xy
    coords.extend([min(x), min(y), max(x), max(y)])

# Create the Extent object that defines the geographic boundaries for analysis
extent = arcpy.Extent(*coords)
print(extent)
# Set the extent as the current environment extent
arcpy.env.extent = extent

# Define paths to various input datasets used in the model
param0=r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\Jaxa_israel_Clip"    # Digital elevation model
param1=r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\slope"               # Slope data
param2= 500                                                            # Buffer distance parameter
param3 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\rain_mean"         # Average rainfall data
param4 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\Veg_values"        # Vegetation data
param5 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\p_water"           # Proximity to water sources
param6 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\aspect"            # Terrain aspect (direction)
param7 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\kadesh_barnea"     # Reference location
param8 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\agri_soils"        # Agricultural soil data
param9=r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\cisterns"            # Water storage locations

# Path to a blank raster template for output
param10=r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\blank_YEAR"         # Empty raster for results
param11 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\Veg_values"       # Vegetation data (repeated)
param12 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\Flow_Direct"      # Water flow direction
param13 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\Watershed_un_p"   # Watershed boundaries
param14 = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\Watersh_Agri_soil" # Watershed with agricultural soils

# Define coordinate system (ITM - Israel Transverse Mercator)
itm_spatial_reference = arcpy.SpatialReference(2039)
arcpy.env.outputCoordinateSystem = itm_spatial_reference

# Set directory paths for outputs and temporary files
base_directory = r"C:\Users\Owner\Documents\Itay\years.gdb"
base_dir = r"C:\Users\Owner\Documents\Itay\years.gdb"
dump_directory = r"C:\Users\Owner\Documents\Itay\Prj_25.gdb"

# Load rainfall data from CSV file
rain_data=pd.read_csv(r"C:\Users\Owner\Documents\Itay\all_by_S.csv", encoding='Windows-1255')
rp=r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\rain_points_2023"

# Define utility functions for raster data conversion
def rasterlist_to_str(rasterlist):
    """Convert a list of raster paths to strings for storage/display"""
    ys_output=[]
    for i in range(len(rasterlist)):
        l = []
        for j in range(len(rasterlist[i])):
            l.append(str(rasterlist[i][j]))
        ys_output.append(l)
    return ys_output

def rasterlist_to_life(rasterlist):
    """Convert a list of raster paths to actual raster objects for processing"""
    ys_output=[]
    for i in range(len(rasterlist)):
        l = []
        for j in range(len(rasterlist[i])):
            raster_path = rasterlist[i][j]    
            l.append(arcpy.Raster(raster_path))
        ys_output.append(l)
    return ys_output

# Determine cell size for the simulation
# Original cell size
ras = Raster(param10)
orig_cell_size = ras.meanCellWidth  # Assuming square cells, else also check meanCellHeight
print(f"Original cell size: {orig_cell_size} m")

# New cell size (larger for simulation efficiency)
new_cell_size = 250

# Calculate scale factor for resampling
scale_factor = new_cell_size / orig_cell_size

# Original dimensions of the raster
orig_cols = ras.width
orig_rows = ras.height
print(f"Original dimensions: {orig_rows} rows, {orig_cols} columns")

# Calculate new dimensions after resampling
new_cols = math.ceil(orig_cols / scale_factor)
new_rows = math.ceil(orig_rows / scale_factor)

print(f"New dimensions: {new_rows} rows, {new_cols} columns")

# Permanent factors

***Permanent Human and Environmental Factors***

This section loads the key environmental factors that remain constant throughout the simulation. 

1. **Agricultural Soil distance**: 
   - Identifies areas with higher proximity to soil types suitable for ancient agricultural practices
   - Higher values indicate better agricultural potential

2. **Distance to Kadesh Barnea**:
   - Measures proximity to a Tell el-Gudeirat (biblical Kadesh Barnea)
   - hypothsized to influence spatial patterns based it being an important hub for the agents at play.

3. **Terrain Aspect Suitability**:
   - Represents the direction that slopes face (north, south, east, west)
   - Affects sun exposure, which influences encampment desirability

4. **Distance to Permanent Water Sources**:
   - Proximity to reliable water sources like springs and wells
   - Critical factor for encampment viability

5. **Vegetation Pasture Suitability**:
   - Indicates areas with vegetation types favorable for pasture
   - Reflects natural resource availability

6. **Mean Annual Rainfall Suitability**:
   - Areas with better precipitation regime (for pasture and agriculture) are graded higher
   - Key determinant of encampment sustainability?

7. **Slope Classification and Suitability**:
   - Categorizes terrain by steepness
   - Identifies areas with slopes appropriate for building structures



In [ ]:
#Import Permanent rasters
l=[r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\fit_Dist_ags',             # Agricultural soil distance
r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\fit_Dist_kadeshb',           # Distance to Kadesh Barnea
r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\aspect_raster_fit',          # Terrain aspect suitability
r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\fit_dist_perwater',          # Distance to permanent water sources
r"C:\Users\Owner\Documents\Itay\Prj_25.gdb\veg_fit",                    # Vegetation suitability
r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\annual_rain_fit',            # Mean Annual rainfall suitability
r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\slope_range',                # Slope range classification
r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\slope_suitability']          # Slope suitability for occupation

# Function to convert a list of raster paths to actual raster objects
def rasterlist_to_life1(rasterlist):
   """Load a list of raster paths into ArcPy raster objects"""
   output=[]
   for i in range(len(rasterlist)):
       raster_path = rasterlist[i]    
       output.append(arcpy.Raster(raster_path))
       
   return output

# Load all permanent factors as raster objects
permanent_results=rasterlist_to_life1(l)

# Unpack the rasters into named variables for easier reference
agri_raster, kb_suitability_raster, aspect_raster, pw_suitability_raster, veg_fit, rain_suitability_raster, rslp_suitability_raster, slp_suitability_raster = permanent_results

# Yearly factors

***Yearly Environmental and Anthropogenic Factors***

This section calculates how environmental and human-related factors change from year to year. The code handles several important aspects:

1. **Resource Pressure Over Time**: The code tracks how resources are used and depleted over multiple years, with a memory effect where recent impacts are weighted more heavily than older ones.

2. **Human Stress Zones**: The model identifies areas that have been heavily impacted by human activity and creates a gradient of "stress" that influences future settlement decisions.

3. **Data Persistence**: The model can load previous simulation results and convert them into usable data objects, allowing for continuous modeling across multiple sessions.

The function uses fuzzy logic (a mathematical approach that allows for degrees of truth rather than simple yes/no values) to create realistic gradients of impact. This helps model the way nomadic groups might avoid overused areas while still considering them as potential future settlement locations once they've had time to recover.

## funcs

In [ ]:
def Yi_params(param10, i, all_outputs):
    """
    This function calculates yearly environmental and anthropogenic factors.
    It tracks resource pressure over time and generates stress zones for modeling.
    
    Parameters:
    - param10: Initial resource raster for the first year
    - i: Current year index 
    - all_outputs: List of outputs from previous years
    
    Returns:
    - return_ras: Raster showing likelihood of returning to previously used locations
    - yi_hu_stressed_z: Raster showing human-stressed zones
    - past_years: NumPy array of accumulated past impacts
    """
     
    # Calculate inter-annual pressure on resources
    if i>0:
        # Initialize an empty array with the same shape as the last year's output
        past_years = np.zeros_like(all_outputs[-1])
        past_years = past_years.reshape(new_rows, new_cols)

        
        # Iterate over all previous years' outputs to calculate cumulative effects
        for idx, output in enumerate(all_outputs):
            # Calculate weights that diminish with time (older years have less influence)
            # Calculate weights that increase with recency
            # Use reversed index to make recent years have more influence
            reversed_idx = len(all_outputs) - 1 - idx  # This makes older years have larger idx values

            decay_factor = 0.7  # Adjust as needed: higher values = faster decay
            weight = (decay_factor ** reversed_idx) * 1  # Standard weight for low impact areas
            weight2 = (decay_factor ** reversed_idx) * 1.5  # Higher weight for high impact areas

            # Apply different weights based on impact intensity
            mask = output < 2           # Areas with low impact
            mask2 = output >= 2         # Areas with high impact
            
            # Accumulate weighted values for different impact zones
            past_years[mask] += weight * output[mask]
            past_years[mask2] += weight2 * output[mask2]
            


        radius_cells = 20  # Convert map units to cells 20 cells*250m cell_size=5km radius
        # Create a grid of coordinates
        y, x = np.ogrid[-radius_cells:radius_cells + 1, -radius_cells:radius_cells + 1]
        # Calculate distances from center
        dist = np.sqrt(x * x + y * y)
        # Create inverse distance weights (with power parameter)
        power = 2  # Adjust as needed: higher values = faster decay with distance
        weights = 1.0 / np.power(dist + 1, power)  # Adding 1 to avoid division by zero

        # Normalize weights to sum to 1
        weights_normalized = weights / np.sum(weights)

        # Apply the convolution
        idw_result = convolve2d(past_years, weights_normalized, mode='same', boundary='fill', fillvalue=0)

        # Convert to raster
        zones = arcpy.NumPyArrayToRaster(idw_result, arcpy.Point(extent.XMin, extent.YMin),
                                      extent.width / all_outputs[i-1].shape[1],
                                      extent.height / all_outputs[i-1].shape[0])
        
        
    else:
        # For the first year, use the initial parameters provided
        last_year_output = Raster(param10)
        past_years=np.zeros((318,250))
        zones=last_year_output

    mean_value = arcpy.management.GetRasterProperties(zones, 'MEAN')
    
    # Handle different scenarios for stress zone calculation
    if float(mean_value.getOutput(0)) == 0:
        # If mean value is zero, assign a baseline value of 10 to all zones
        yi_hu_stressed_z = zones + 10
    else:
        # Remove zero values from consideration
        nz = arcpy.sa.SetNull(zones, zones, "VALUE = 0")
        z = Raster(nz) 
        # Save the intermediate raster
        #z.save(os.path.join(dump_directory, 'zone'))
        
        # Get the range of values in the stress zones
        hs_raster_obj = Raster(z)
        hs_max_value = arcpy.management.GetRasterProperties(hs_raster_obj, 'MAXIMUM').getOutput(0)
        hs_min_value = arcpy.management.GetRasterProperties(hs_raster_obj, 'MINIMUM').getOutput(0)
        hs_max_value = float(hs_max_value)
        hs_min_value = float(hs_min_value)
        
        # Apply fuzzy logic to create a gradient of stress levels
        hs_FuzzyAlgorithm = FuzzyLinear(2,0)
        hsFuzzyMember = FuzzyMembership(z, hs_FuzzyAlgorithm)
        yi_hu_stressed_z = hsFuzzyMember * 10
        # Fill null values with baseline value of 10
        yi_hu_stressed_z = arcpy.sa.Con(IsNull(yi_hu_stressed_z), 10, yi_hu_stressed_z)
    
    # Save the human stressed zones raster for this year
    yi_hu_stressed_z.save(os.path.join(base_directory, 'Y{}_hu_stressed_zones'.format(i)))

    
    # Calculate likelihood to return to previously used locations
    return_arr = past_years * 5
    return_ras=arcpy.NumPyArrayToRaster(return_arr, arcpy.Point(extent.XMin, extent.YMin),
                                      extent.width / return_arr.shape[1],
                                      extent.height / return_arr.shape[0])
    return_ras.save(os.path.join(base_directory, 'Y{}_return_ras'.format(i)))
    
    return return_ras, yi_hu_stressed_z, past_years

## load yearly maps

In [ ]:
# Load previously saved outputs from a pickle file
y1_outputs = None
file_path = r"C:\Users\Owner\Documents\Itay\y_outputs.pkl"
with open(file_path, 'rb') as f:
    y1_outputs = pickle.load(f)

y1_outputs

def rasterlist_to_life(rasterlist):
    """
    Converts a list of raster file paths into actual ArcPy Raster objects.
    
    Parameters:
    - rasterlist: A nested list of raster file paths
    
    Returns:
    - ys_output: A nested list of ArcPy Raster objects
    """
    ys_output = []
    for i in range(len(rasterlist)):
        l = []
        for j in range(len(rasterlist[i])):
            raster_path = y1_outputs[i][j]    
            l.append(arcpy.Raster(raster_path))
        ys_output.append(l)
    return ys_output

# Convert the list of raster paths to actual Raster objects
y_output = rasterlist_to_life(y1_outputs)

# Summerize

***Summarize: Weighted Sum Calculations for Permanent and Yearly Factors***

This section defines two functions that calculate weighted sums for permanent and yearly factors, which are key in determining the suitability of land for agro-pastoral nomadic societies based on multiple environmental and anthropological factors.

1. **`summarize_outputs_p`**:
   - This function calculates the weighted sum of permanent factors using predefined weights (`wp1` to `wp8`) for different environmental suitability factors.
   - The weighted sum incorporates factors such as proximity to arable land, terrain suitability, water sources, vegetation, rainfall, and slope suitability.
   - The result is a composite score (a raster) representing the overall suitability for permanent settlement or activities.


2. **`summarize_outputs_y`**:
   - This function calculates the weighted sum of yearly factors using similar principles as the first function, but for factors that may change over time (like yearly land productivity, water availability, etc.).
   - It uses weights (`ws1` to `ws6`) for yearly suitability factors and combines them with an additional factor (`ws_p`) for final suitability.
   - The output, `Yi_suitability`, is a raster layer that represents yearly suitability, and the result is saved with a unique filename based on the iteration (`i`).
   


In [ ]:
#Sum weights
def summarize_outputs_p(wp2,wp4,wp6,wp8):
    # Calculate the weighted sum for permanent outputs
    weighted_sum_permanent = ((wp2*Raster(kb_suitability_raster))+
                               (wp4*Raster(pw_suitability_raster))+
                               (wp6*Raster(rain_suitability_raster))
                              +(wp8*Raster(slp_suitability_raster)))

    
    return weighted_sum_permanent
def summarize_outputs_y(ws1, ws2, ws6, ws_p,return_ras,
                        yi_hu_stressed_z, YrainFuzzyMember_calc,i):    
    # Calculate the weighted sum for yearly outputs
    weighted_sum_yearly = ((ws1*Raster(return_ras))+(ws2*Raster(yi_hu_stressed_z))+
                            (ws6*Raster(YrainFuzzyMember_calc)))

    # Optionally, you can perform additional processing or output creation using arcpy if needed.
    Yi_suitability=weighted_sum_yearly+ws_p
    Yi_suitability.save(os.path.join(base_directory, 'Y{}_suitability_ras'.format(i)))
    return Yi_suitability

# Agents

## Agents funcs

In [ ]:
# Function to calculate the direction in which the agent will grow or expand.
def grow_direction(agent, r):     
    # Get current agent's position (x, y)
    x, y = agent.pos
    pg = []  # List to store potential growth cells and their suitability values

    # Get the neighborhood of the agent with the given radius (r)
    nbr = agent.model.grid.get_neighborhood(agent.pos, moore=True, include_center=False, radius=r)
    
    # Get the mean environmental value of the agent's current position with the specified radius
    mean_val = env_mean_val(agent.model, agent.pos, r)
    
    # Iterate through neighboring cells to evaluate suitability
    for cell in nbr:
        xi, yi = cell
        # Get the suitability value of the current cell
        suit_val = agent.model.suitability_raster[to_numpy_y(agent.model, yi), xi]
        
        # If the cell has a higher suitability than the mean, add it to potential growth list
        if suit_val > mean_val:
            pg.append([(xi, yi), suit_val])
    
    # Return the list of potential growth cells
    return pg

# Function to calculate the weight for each cell based on its suitability value
def get_weight(agent, element):
    cell, suit_val = element
    # Ensure a minimum weight to avoid division by zero
    return max(suit_val, 0.0001)

# Function to make the agent grow based on its suitability and environment
def grow(model, agent, r):
    # Get agent's current position (x, y)
    x, y = agent.pos
    
    # Calculate the growth radius as one-third of the specified radius
    grow_rad = int(r / 3)
    
    # Get the neighboring cells around the agent within the growth radius
    nbr = agent.model.grid.get_neighborhood(agent.pos, moore=True, include_center=False, radius=grow_rad)
    
    # Get the total manpower available for growth
    tot_mnpw = agent.manpower
    
    # Calculate the mean environmental value for the agent's position
    mean_val = env_mean_val(model, agent.pos, r)
    
    # Get potential growth directions for the agent
    pg = grow_direction(agent, grow_rad)
    
    # Calculate weights for the potential growth cells
    weights = [get_weight(agent, element) for element in pg]
    
    # Randomly choose growth cells based on weights, with up to 3 cells selected
    if len(pg) > 3:
        ch_cell = random.choices(pg, weights=weights, k=3)
    else:
        ln = len(pg) - 1
        ch_cell = random.choices(pg, weights=weights, k=ln)
    
    # Apply growth to the selected cells in the target raster
    for i in ch_cell:
        xn, yn = i[0]
        agent.model.target_raster[to_numpy_y(agent.model, yn), xn] += 0.333
    
    # Decrease surplus and manpower after growth
    agent.surplus -= 1
    agent.manpower -= 1
    
    # Degrade the environment due to the agent's growth
    env_degrade(agent.model, xn, yn, grow_rad, 0.2, 0.3)

# Function to set the agent's camp (stop moving and establish a position)
def set_camp(agent):
    # Get agent's current position (x, y)
    x, y = agent.pos
    
    # Decrease surplus and manpower once camp is set
    agent.surplus -= 1
    agent.manpower -= 1
    
    # Mark the cell in the target raster as the camp location
    agent.model.target_raster[to_numpy_y(agent.model, y), x] += 0.75

# Optimized version of the function to move the agent locally based on environmental conditions
def move_local(model, agent, env_r, search_r, territory):
    """
    Optimized version of move_local that reduces redundant calculations
    and improves performance for frequent calls.
    
    Args:
        model: The model instance
        agent: The agent to move
        env_r: Environment check radius
        search_r: Search radius for new positions
        territory: Set/list of valid territory positions
    
    Returns:
        New position tuple if move is possible, False otherwise
    """
    # Get current position of the agent
    current_pos = agent.pos
    
    # Calculate the mean environmental value for the current position
    mean_env_value = env_mean_val(model, current_pos, env_r)
    
    # If the environment value is low, attempt to move
    if mean_env_value <= 5:
        # Get neighboring cells around the agent for movement options
        near_env = model.grid.get_neighborhood(current_pos, moore=True, include_center=False, radius=search_r)
        
        # Convert territory to a set for faster lookups
        territory_set = set(territory)
        
        # Pre-calculate occupied positions for faster lookup
        occupied_positions = set()
        for pos in near_env:
            if model.grid.get_cell_list_contents(pos):
                occupied_positions.add(pos)
        
        # Calculate a target threshold for movement
        target_threshold = mean_env_value + 0.5
        
        # Pre-calculate valid positions for movement based on environmental suitability
        valid_positions = []
        for pos in near_env:
            if (pos in territory_set and 
                pos not in occupied_positions and 
                model.place_raster[to_numpy_y(model, pos[1]), pos[0]] == 1):  # Check if position is allowed
                pos_env_value = env_mean_val(model, pos, env_r)
                if pos_env_value > target_threshold:
                    valid_positions.append(pos)
        
        # Choose a random valid position for the agent to move to
        if valid_positions:
            return random.choice(valid_positions)
    
    # Return False if no valid position is found
    return False

# Function to calculate the number of agents based on total area and carrying capacity
def Num_agents():
    total_area_sq_km = 3251.54  # The Negev Highlands (model 2024)
    
    # Calculate the maximum number of agents that the area can support
    Max_carry_agents = total_area_sq_km / 18  # 9 km² per member agent
    
    # Calculate the number of agents based on the carrying capacity
    carry_agents = Max_carry_agents / 20  # Max_carry_agents * 0.5
    
    # Return the number of agents as an integer
    return int(np.floor(carry_agents))

# Function to calculate the pasture suitability value based on environmental factors
def pasture_val(model, current_pos, r):
    # Get the neighborhood of the current position
    nbr = model.grid.get_neighborhood(current_pos, moore=True, include_center=False, radius=r)
    
    # Filter out any cells that are out of bounds or outside the region of interest
    nbr = [cell for cell in nbr if not model.grid.out_of_bounds(cell) and cell[0] < 280]
    
    # Avoid division by zero in the stress raster by replacing zeros with a small value
    stress_safe = np.where(model.stress_ras == 0, 0.01, model.stress_ras)
    
    # Calculate the inverse stress factor (higher stress means less suitable)
    inverse_stress = 1 / (stress_safe / 10)
    
    # Cap the stress value to a maximum of 9
    capped_stress = np.minimum((inverse_stress - 1), 9)
    
    # Calculate the final vegetation map by adjusting for stress
    veg_map = np.maximum((model.veg_ras * (1 - (capped_stress * 0.1))), 0)
    
    # Collect the environmental values of neighboring cells
    env_values = [
        veg_map[to_numpy_y(model, pos[1]), np.clip(pos[0], 0, 279)]  # Vegetation suitability
        for pos in nbr
    ]
    
    # Calculate the mean environmental value of the neighborhood
    tot_env_value = sum(env_values) 
    
    # Return the mean environmental value
    return tot_env_value

# Function to resample a raster to a new resolution (cell size)
def resample(raster, path):
    # Resample the raster to the desired cell size using Bilinear resampling
    arcpy.management.Resample(in_raster=raster, out_raster=path, cell_size=250, resampling_type="Bilinear")
    
    # Return the resampled raster object
    resampled_raster_obj = arcpy.Raster(path)
    return resampled_raster_obj


## Agents classes

### **Household Agent**

**Household Agent**

This class represents a nomadic household unit in the simulation.
 
**Key Attributes**
- **Surplus**: The household's available energy and resources.
- **Manpower**: The number of active members contributing to household tasks.
- **Memory**: Tracks past encampments and enclosures.
- **Territory**: The area occupied and influenced by the household.
 
**Key Behaviors**
1. **Step Execution**
   - Updates household surplus and manpower.
   - Manages upkeep costs for members.
   - Decides whether to reuse or build new enclosures.
 
2. **Year Initiation**
   - Places the household in the environment.
   - Updates memory and environmental impact.
 
3. **Enclosure Management**
   - Finds, maintains, or builds enclosures based on resources and environmental conditions.


In [ ]:
class Household_Agent(mesa.Agent):
    """
    Represents a household unit in the model.
    Each household manages resources, manpower, territory, and enclosures over time.
   
    Attributes:
    ----------
    flock_head : int
        Represents the livestock units available to the household.
    surplus : int
        Represents the resource surplus of the household.
    manpower : int
        The number of humans in the household, representing its workforce or
        population.
    territory : list
        The territory currently occupied by the household. Can be used to
        track spatial information like coordinates or grid cells.
    memory : list
    """

    def __init__(self, model, threshold, manpower=None, surplus=None, flock_head=None):
        super().__init__(model)  # Initialize the agent within the Mesa framework

        # Core attributes of the household
        if flock_head is None:
            self.flock_head = 0
            for _ in range(10):
                # ~18 livestock units per person, ~6 people per family, adapted from finkelstein and rosen
                flock_n = np.floor(random.gauss(105, 15))
                self.flock_head += flock_n
        else:
            self.flock_head = flock_head

        if manpower is None:
            self.manpower = max(35, np.floor(self.flock_head / 20))
        else:
            self.manpower = manpower

        if surplus is None:
            self.surplus = 0
        else:
            self.surplus = surplus
        self.territory = None  # Area claimed by the household
        self.memory = []  # Record of past encampments
        self.enc_memory = []  # Record of past enclosures
        #self.members = members  # List of members 
        self.threshold = threshold  # Threshold for resource-based decisions =2
        self.own_suitability_raster=None
                          

    def step(self):
        """Defines the actions the household takes in a simulation step."""
        # Update household's total surplus by summing member surpluses
        self.calc_surplus()
        
        # Simplified scenario system (bad, neutral, good) integrated with communal costs
        scenario_roll = random.random()
        
        # Base communal cost calculation
        base_communal_cost = math.ceil(self.manpower * 0.3)
        
        if scenario_roll < 0.25:  # Bad scenario (25% chance)
            # Higher costs due to disease, conflict, harsh weather, etc.
            scenario_multiplier = random.uniform(1.3, 1.6)
            communal_cost = math.ceil(base_communal_cost * scenario_multiplier)
            
            # Bad scenarios can also affect livestock
            if random.random() < 0.6:  # 60% chance of affecting livestock
                self.flock_head = math.floor(self.flock_head * (1 - random.uniform(0.05, 0.12)))
                
        elif scenario_roll < 0.75:  # Neutral scenario (50% chance)
            # Normal costs
            scenario_multiplier = random.uniform(0.95, 1.05)
            communal_cost = math.ceil(base_communal_cost * scenario_multiplier)
            
        else:  # Good scenario (25% chance)
            # Lower costs due to favorable conditions, gifts, good trade
            scenario_multiplier = random.uniform(0.7, 0.9)
            communal_cost = math.ceil(base_communal_cost * scenario_multiplier)
            
            # Good scenarios might bring additional benefits
            if random.random() < 0.5:  # 50% chance of bonus
                # Small surplus boost or manpower increase
                if random.random() < 0.7:
                    self.surplus += base_communal_cost * random.uniform(0.1, 0.3)
                else:
                    self.manpower += random.randint(1, 2)
        
        # Apply communal costs with more impact for larger surpluses
        # As surplus grows, costs increase more than linearly
        if self.surplus > 50:
            # Additional social obligations for wealthy households (feasts, gifts, etc.)
            wealth_factor = 1 + (self.surplus - 50) / 200  # Increases with surplus
            communal_cost = math.ceil(communal_cost * wealth_factor)
        
        # Apply the communal cost
        if self.surplus > communal_cost:
            self.surplus -= communal_cost
        else:
            # If not enough surplus, reduce what we can and potentially impact other resources
            self.surplus = 0
            deficit = communal_cost - self.surplus
            
            # Deficit has some impact on manpower or flock
            if random.random() < 0.3:
                self.manpower = max(1, self.manpower - math.ceil(deficit / 20))
            else:
                self.flock_head = max(50, self.flock_head - deficit)
        self.manpower-=10 #for yearly movement
        herders=(self.flock_head//75)
        self.manpower-=herders #manpower for herding
        # Check for an existing enclosure within the territory
        existing_enclosure = self.find_recent_enclosure()

        # Environmental quality assessment
        current_env_value = env_mean_val(self.model, self.pos, 20)
        env_quality = current_env_value / 10.0  # Normalized to 0-1 range
        # Calculate surplus ratio as prosperity indicator
        surplus_ratio = self.surplus / (self.manpower+10+herders) 
        # Calculate household prosperity index
        #print('env_qulity:',env_quality, 'surplus_ratio:', surplus_ratio)
        #prosperity_index = (surplus_ratio / (self.threshold * 2)) * env_quality
        prosperity_index = surplus_ratio  * env_quality
        is_prosperous = prosperity_index > 0.7
        if is_prosperous:
            # Only proceed with enclosure-related actions if manpower and surplus meet thresholds
            if self.manpower >= 15 and self.surplus >= self.threshold:
                if existing_enclosure:
                    
                    xn, yn = existing_enclosure
                    
                    # Basic cost for reuse
                    manpower_cost = 10
                    surplus_cost = 6
                    
                    #print(f"Enclosure reused by agent: {self.unique_id}, prosperity: {prosperity_index:.2f}")
                    
                    # Now reduce household resources
                    self.manpower = max(self.manpower - manpower_cost, 0)
                    self.surplus = max(self.surplus - surplus_cost, 0)
                    self.model.target_raster[to_numpy_y(self.model, yn), xn] += 1
                    self.model.enclosures.append([existing_enclosure, self.model.year, self])
                    self.enc_memory.append([existing_enclosure, self.model.year])
                    self.manpower += manpower_cost * 0.9  # Partial recovery of workforce
                
                elif prosperity_index>0.8 and self.manpower >= 40 and self.surplus >= self.threshold * 1.5:
                    # Prosperous households can build new enclosures
                    enc_pos = self.build_enclosure(self.territory, current_env_value)
                    xn, yn = enc_pos
                    
                    # Higher investment by prosperous households
                    manpower_cost = 20
                    surplus_cost = 25
                    
                    # If not enough surplus, convert flock to surplus (emergency measure)
                    if self.surplus < surplus_cost:
                        surplus_needed = surplus_cost - self.surplus
                        flock_needed_for_surplus = surplus_needed * 3
                        manpower_reserve = self.manpower * 20                    
                        available_flock = max(0, self.flock_head - manpower_reserve)                    
                        flock_to_convert = min(available_flock, flock_needed_for_surplus)                    
                        
                        if flock_to_convert > 0:
                            self.flock_head -= flock_to_convert
                            self.surplus += flock_to_convert / 3
                    
                    #print(f"New enclosure built by: {self.unique_id}, prosperity: {prosperity_index:.2f}")
                    
                    # More investment leads to better quality enclosures
                    enclosure_quality = 2 + (prosperity_index * 2)  # Higher prosperity = more activity
                    
                    self.manpower = max(self.manpower - manpower_cost, 0)
                    self.surplus = max(self.surplus - surplus_cost, 0)
                    self.model.target_raster[to_numpy_y(self.model, yn), xn] += enclosure_quality
                    self.model.enclosures.append([enc_pos, self.model.year, self])
                    self.enc_memory.append([enc_pos, self.model.year])
                    self.manpower += manpower_cost * 0.9  # Partial recovery of workforce
        # Handle survival crisis if surplus is too low
        if self.surplus < 10:
            survived = self.handle_survival_crisis(threshold=10)
            if not survived:
                self.remove()
                return  # Exit the step method if household is removed
        # Update flock size based on environmental conditions
        self.update_flock_size()

        # Adjust manpower based on random chance
        self.manpower = max(1, self.manpower + 10)
        self.manpower+=herders #manpower for herding
        if random.uniform(0, 1) > 0.8:
            self.manpower += math.ceil(5*random.uniform(0,1))
        if self.manpower > 35 and random.uniform(0, 1) > 0.8:
            self.manpower -= math.ceil(5*random.uniform(0,1))
        # Surplus-dependent manpower bonuses
        if self.surplus > 100:
            self.manpower += min(5, self.surplus // 20)
        # Surplus decay over time
        if self.surplus > 50:
            base_decay_rate = 0.05  # 5% base decay
            # Increased decay rate for higher surplus
            scaling_factor = 0.003 * (self.surplus - 50)
            decay_rate = min(0.4, base_decay_rate + scaling_factor)  # Cap at 40%
            decay_amount = self.surplus * decay_rate
            self.surplus -= decay_amount

    def year_initiation(self):
        """Initial setup for the household at the start of the year."""
        self.own_suitability_raster=None
        self.memory = [i for i in self.memory if (self.model.year - i[2]) * random.random() < 5]


        mesa_x, mesa_y = place_household(self.model, self, self.model.suitability_raster)
        self.model.grid.place_agent(self, (mesa_x, mesa_y))
         # Check environmental conditions
        current_env_value = env_mean_val(self.model, self.pos, 20)

        # Calculate territory radius based on environmental conditions
        base_radius = 20  # Default radius
        # Expand territory in poor environmental conditions
        if current_env_value < 3.5:  # Poor conditions
            territory_radius = base_radius + 10  # Significantly larger territory
        elif current_env_value < 5.0:  # Moderate conditions
            territory_radius = base_radius + 5   # Somewhat larger territory
        else:  # Good conditions
            territory_radius = base_radius       # Standard territory size
        self.territory = self.model.grid.get_neighborhood((mesa_x, mesa_y), moore=True, include_center=True, radius=territory_radius)
        self.model.territories.append(self.territory)
        set_camp(self)
        
        for _ in range (9):
            selected_cell = place_members(self.model, self, self.territory)  # Place member
            env_degrade(self.model, selected_cell[0], selected_cell[1], 11, 0.3, 0.5)  # Simulate member degradation
            self.memory.append([selected_cell, env_mean_val(self.model, selected_cell, 5), self.model.year])
            self.model.target_raster[to_numpy_y(self.model, selected_cell[1]), selected_cell[0]] += 0.5
        
        env_degrade(self.model, mesa_x, mesa_y, territory_radius, 0.5, 0.9)
        self.memory.append([self.pos, env_mean_val(self.model, self.pos, territory_radius), self.model.year])
    def calc_surplus(self):
        """
        Calculates resource surplus based on manpower, herd size, agricultural potential, 
        and environmental stress conditions, ensuring realistic consumption and limits.
        """
        # Reset surplus at beginning of calculation to avoid accumulation
        # This is a major change - calculate actual surplus each time instead of incrementing
        current_surplus = self.surplus
        self.surplus = 0
        
        # Check agricultural potential in territory
        ag_values = []
        for cell in self.territory:
            x, y = cell
            ag_values.append(self.model.ag_ras[to_numpy_y(self.model, y), x])
        
        # Calculate mean agricultural value in territory (0-10 scale)
        mean_ag_value = sum(ag_values) / len(ag_values) if ag_values else 0
        
        # Reduce livestock need based on agricultural potential
        ag_benefit = min(0.4, mean_ag_value * 0.04)  # Up to 40% reduction
        
        # Adjust base livestock need according to agricultural potential
        base_need_per_person = 18 * (1 - ag_benefit)
        subsistence_need = self.manpower * base_need_per_person
        
        # Calculate environmental carrying capacity using pasture_val
        env_value = pasture_val(self.model, self.pos, 25)
        neighborhood = self.model.grid.get_neighborhood(self.pos, moore=True, include_center=False, radius=25)
        neighborhood_size = len(neighborhood)
        avg_env_value = env_value / neighborhood_size if neighborhood_size > 0 else 0
        
        # Medium quality (5 on 0-10 scale) with 250m*250m cells (0.0625 sq km)
        # Each cell at medium quality supports 1.125 goats
        quality_factor = avg_env_value / 5.0
        carrying_capacity = neighborhood_size * 1.125 * quality_factor
        
        # Calculate stress ratio based on carrying capacity
        stress_ratio = max(0, 1 - (carrying_capacity / self.flock_head)) if self.flock_head > 0 else 0
        
        # Calculate costs with environmental influence
        maintenance_cost = self.flock_head * 0.05  # Base maintenance cost
        stress_cost = self.flock_head * stress_ratio * 0.15  # Higher penalty for overstocking
        
        total_cost = math.ceil(subsistence_need + maintenance_cost + stress_cost)
        
        # Production is excess livestock after essential needs are covered
        excess_livestock = max(0, self.flock_head - total_cost)
        
        # Convert excess livestock to surplus value
        # Based on the Gunther et al. 2021 and Dhal and Hjort 1976:96 :
        # Consider age/sex structure of the excess livestock
        juvenile_males_ratio = 0.148  # Approximation based on herd demographics in d&h
        adult_females_ratio = 0.579
        adult_males_ratio = 0.088
        juvenile_females_ratio = 0.185
        
        # Different surplus value per type (reflecting their value as described in Gunther et al. 2021)
        juvenile_males_value = 0.3  # First to be culled, lowest value
        adult_females_value = 0.2   # Breeding stock, rarely culled
        adult_males_value = 0.4     # Higher meat value but less reproductive value
        juvenile_females_value = 0.25 # Future breeding stock
        
        # Calculate surplus from different components of the herd
        surplus_from_juvenile_males = excess_livestock * juvenile_males_ratio * juvenile_males_value
        surplus_from_adult_males = excess_livestock * adult_males_ratio * adult_males_value
        surplus_from_adult_females = excess_livestock * adult_females_ratio * adult_females_value
        surplus_from_juvenile_females = excess_livestock * juvenile_females_ratio * juvenile_females_value
        
        # Prioritize culling juvenile males first, then adult males, then others
        culling_priority = [
            (juvenile_males_ratio, surplus_from_juvenile_males),
            (adult_males_ratio, surplus_from_adult_males),
            (adult_females_ratio, surplus_from_adult_females),
            (juvenile_females_ratio, surplus_from_juvenile_females)
        ]
        
        total_surplus = 0
        remaining_excess = excess_livestock
        
        # Use the culling priority to determine surplus generation
        for ratio, value in culling_priority:
            # How many of this type can be culled
            available = math.floor(self.flock_head * ratio)
            to_cull = min(remaining_excess, available)
            
            if to_cull > 0:
                # Calculate value based on type-specific value
                type_value_per_head = value / (excess_livestock * ratio) if excess_livestock * ratio > 0 else 0
                total_surplus += to_cull * type_value_per_head
                remaining_excess -= to_cull
                
            if remaining_excess <= 0:
                break
        
        # Add resource decay - resources spoil or degrade over time
        decay_rate = 0.3  # 30% of surplus decays each time step
        preserved_surplus = current_surplus * (1 - decay_rate)
        
        # Additional basic consumption - households use surplus for non-essential needs
        consumption_per_person = 0.8  # Each person consumes some surplus for quality of life
        additional_consumption = self.manpower * consumption_per_person
        
        # Calculate final surplus
        final_surplus = preserved_surplus + total_surplus - additional_consumption
        
        # Cap the maximum surplus a household can maintain
        max_surplus_cap = 200 + (self.manpower * 5)  # Base cap plus additional per person
        self.surplus = min(max(0, final_surplus), max_surplus_cap)
    # Extract this from your step method and replace with the following:
    def update_flock_size(self):
        """
        Updates flock size based on environmental conditions and stress levels.
        Should be called from the step method.
        """
        # Calculate environmental value
        env_value = pasture_val(self.model, self.pos, 30)
        
         # Get the neighborhood cells
        neighborhood = self.model.grid.get_neighborhood(self.pos, moore=True, include_center=False, radius=30)
        neighborhood_size = len(neighborhood)
        
        # Calculate average environmental value per cell
        avg_env_value = env_value / neighborhood_size if neighborhood_size > 0 else 0
        
        # Environmental quality is on a 0-10 scale
        # Medium quality (5 on 0-10 scale) means 1 sq km can support ~ 18 goats
        # Each cell is 0.0625 sq km (250m × 250m)
        # So a medium quality cell (5) supports 18 * 0.0625 = 1.125 goats
        
        # Scale based on actual quality
        # If avg_env_value is 10 (max quality), it can support twice as many goats as medium quality
        # If avg_env_value is 0, it can support no goats
        quality_factor = avg_env_value / 5.0  # Ratio compared to medium quality
        
        # Each cell's carrying capacity
        goats_per_cell = 1.125 * quality_factor
        
        # Total carrying capacity for the neighborhood
        carrying_capacity = goats_per_cell * neighborhood_size
        
        # Calculate resource-to-livestock ratio
        ratio = carrying_capacity / self.flock_head if self.flock_head > 0 else 2.0
        
        # Get average stress level in territory
        stress_values = []
        for cell in self.territory:
            x, y = cell
            numpy_y = to_numpy_y(self.model, y)
            if 0 <= numpy_y < self.model.stress_ras.shape[0] and 0 <= x < self.model.stress_ras.shape[1]:
                stress_values.append(self.model.stress_ras[numpy_y, x])
        
        avg_stress = sum(stress_values) / len(stress_values) if stress_values else 0
        stress_factor = 1 - min(0.5, avg_stress / 20)  # Higher stress reduces growth
        # Calculate small flock factor - smaller flocks get enhanced recovery
        small_flock_factor = 1.0
        if self.flock_head < 500:
            # Enhanced recovery factor increases as flock size decreases
            small_flock_factor = 2.0 - (self.flock_head / 500)  # Scales from 2.0 to 1.0
        # Adjust flock size based on resource ratio and stress
        if ratio > 1.1:  # More resources than needed
            # Growth factor diminishes with large herds and high stress
            max_growth_rate = 0.1 * stress_factor* small_flock_factor
            # Apply diminishing returns as flock size increases
            diminishing_factor = max(0.3, 1 - (self.flock_head / 1000))
            
            # Calculate actual growth rate with randomization
            growth_rate = max_growth_rate * diminishing_factor * min(ratio, 2.0) * random.uniform(0.7, 1.3)
            
            # Apply growth with a cap based on carrying capacity
            max_sustainable = carrying_capacity * 1.2  # Allow some overshoot
            growth_amount = min(
                self.flock_head * growth_rate,  # Normal growth
                max_sustainable - self.flock_head  # Cap to sustainable level
            )
            
            self.flock_head = np.floor(self.flock_head + max(0, growth_amount))
            
        elif ratio < 0.9:  # Not enough resources
            # Higher decline rates when resources are scarce
            base_decline = 0.1 * (1 + (0.9 - ratio) * 2)
            # More severe decline with higher stress
            stress_impact = 1 + (avg_stress / 20)
            # Reduced decline for small flocks - emergency preservation instinct
            if self.flock_head < 100:
                # Reduce decline rate for small flocks to give them a chance to recover
                base_decline *= max(0.4, self.flock_head / 100)  # Scales from 0.4 to 1.0
            # Calculate actual decline rate with randomization
            decline_rate = base_decline * stress_impact * random.uniform(0.8, 1.2)
            
            # Apply decline
            self.flock_head = np.floor(self.flock_head * (1 - decline_rate))  
    
    def find_recent_enclosure(self):
        """Finds the most recent usable enclosure within the households territory."""
        recent_enclosures = []
        current_year = self.model.year
        
        # Prioritize household's own enclosures
        for enc in self.enc_memory:
            if enc[0] in self.territory and (current_year - enc[1]) < 20:
                score = 20 - (current_year - enc[1])  # Newer enclosures have higher priority
                recent_enclosures.append((enc[0], score))
        
        # Check all enclosures in the model
        for enc in self.model.enclosures:
            if enc[0] in self.territory and (current_year - enc[1]) < 15 and enc[1]!=current_year:
                score = (15 - (current_year - enc[1])) * 0.8  # Lower priority than own enclosures
                recent_enclosures.append((enc[0], score))
        
        if not recent_enclosures:
            return None
        
        # Adjust scores with environmental factors
        enhanced_scores = []
        for pos, score in recent_enclosures:
            x, y = pos
            veg_value = self.model.veg_ras[to_numpy_y(self.model, y), x]
            ag_value = self.model.ag_ras[to_numpy_y(self.model, y), x]
            env_factor = 1.0 + ((veg_value * 0.02) + (ag_value * 0.02))
            final_score = score * env_factor * random.uniform(0.8, 1.2)
            enhanced_scores.append((pos, final_score))
        
        if not enhanced_scores:
            return None
        
        # Select an enclosure probabilistically based on weighted scores
        selected_idx = random.choices(range(len(enhanced_scores)), weights=[s[1] for s in enhanced_scores], k=1)[0]
        return enhanced_scores[selected_idx][0]
    def handle_survival_crisis(self, threshold=10):
        """
        Handles survival conditions when surplus falls below a threshold by strategically
        culling livestock based on demographic categories.
        
        Parameters:
        ----------
        threshold : int
            The surplus threshold below which survival measures are triggered
        
        Returns:
        -------
        bool
            True if the household survives, False if it should be removed
        """
        if self.surplus >= threshold:
            return True
        
        emergency_surplus = 0
        
        # Calculate approximate herd composition based on realistic demographics (dhal and hjort)
        juvenile_males = math.floor(self.flock_head *  0.148)
        adult_males = math.floor(self.flock_head * 0.088)
        juvenile_females = math.floor(self.flock_head * 0.185)
        adult_females = math.floor(self.flock_head * 0.579)
        
        # First strategy: Sacrifice juvenile males
        juvenile_males_to_cull = min(juvenile_males, max(0, math.ceil((100 - self.surplus) / 0.3)))
        if juvenile_males_to_cull > 0:
            self.flock_head -= juvenile_males_to_cull
            emergency_surplus += juvenile_males_to_cull * 0.3
        
        # If still in crisis, sacrifice some adult males
        if self.surplus + emergency_surplus < 50 and adult_males > 0:
            adult_males_to_cull = min(adult_males, max(0, math.ceil((50 - self.surplus - emergency_surplus) / 0.4)))
            self.flock_head -= adult_males_to_cull
            emergency_surplus += adult_males_to_cull * 0.4
        
        # Last resort: Start culling juvenile females
        if self.surplus + emergency_surplus < 25 and juvenile_females > 0:
            max_juvenile_females_to_cull = math.floor(juvenile_females * 0.8)
            juvenile_females_to_cull = min(max_juvenile_females_to_cull, 
                                        max(0, math.ceil((25 - self.surplus - emergency_surplus) / 0.25)))
            self.flock_head -= juvenile_females_to_cull
            emergency_surplus += juvenile_females_to_cull * 0.25
        
        # Absolute dire emergency: Begin culling some adult females
        if self.surplus + emergency_surplus < 10 and adult_females > 0:
            max_adult_females_to_cull = math.floor(adult_females * 0.4)
            adult_females_to_cull = min(max_adult_females_to_cull,
                                    max(0, math.ceil((10 - self.surplus - emergency_surplus) / 0.2)))
            self.flock_head -= adult_females_to_cull
            emergency_surplus += adult_females_to_cull * 0.2
            self.in_severe_crisis = True
        else:
            self.in_severe_crisis = False
        
        # Add the emergency surplus to the household's resources
        self.surplus += emergency_surplus
        
        # Check if the household can survive
        if self.surplus < 0 or self.flock_head < 15 or self.manpower <= 0:
            print('Removing agent', self, 'surplus:', self.surplus, 'flock_head:', self.flock_head, 'manpower:', self.manpower)
            # Store agent resources for new agent creation
            leftover_resources = {
                'manpower': max(1, self.manpower),
                'surplus': max(0, self.surplus),
                'flock_head': max(0, self.flock_head)
            }
            # Add to list of scheduled new agents
            if not hasattr(self.model, 'scheduled_new_agents'):
                self.model.scheduled_new_agents = []

            self.model.scheduled_new_agents.append(leftover_resources)
            return False  # Household should be removed
        
        # Ensure minimum manpower
        self.manpower = max(6, self.manpower)
        
        # If in severe crisis mode, adjust behavior
        if self.in_severe_crisis:
            # Reduce territory range for conservative grazing
            self.territory = self.model.grid.get_neighborhood(self.pos, moore=True, include_center=True, radius=15)
        
        return True  # Household survives
    def build_enclosure(self, nbr, mean_val):
        """
        Selects the best cell for building an enclosure using probabilistic methods
        that take into account environmental factors and existing structures.
        """
        bst_cells = []
        
        # First gather all possible options with their base scores
        
        # Recent own enclosures with higher weight
        for enc in self.enc_memory:
            if enc[0] in nbr:
                age = self.model.year - enc[1]
                xi, yi = enc[0]
                
                # Get environmental values
                veg_value = self.model.veg_ras[to_numpy_y(self.model, yi), xi]
                ag_value = self.model.ag_ras[to_numpy_y(self.model, yi), xi]
                suit_val = self.model.suitability_raster[to_numpy_y(self.model, yi), xi]
                
                # Calculate base score
                if age < 15:  # Recent enclosures
                    base_score = 20 - age
                else:  # Older enclosures
                    base_score = 5 + (20 - min(age, 20))/3
                    
                # Apply environmental factors
                env_factor = 1.0 + (veg_value * 0.02) + (ag_value * 0.02) + (suit_val * 0.02)
                total_score = base_score * env_factor
                
                # Apply ownership bias (own enclosures are preferred)
                ownership_bias = 1.5
                final_score = total_score * ownership_bias
                
                # Add randomization for probabilistic selection (±10%)
                random_factor = random.uniform(0.9, 1.1)
                bst_cells.append([enc[0], final_score * random_factor])
        
        # Consider other enclosures in the simulation
        for enc_data in self.model.enclosures:
            enc_pos, enc_year, enc_owner = enc_data
            if enc_pos in nbr and enc_pos not in [e[0] for e in self.enc_memory]:
                age = self.model.year - enc_year
                if age > 15:  # Only consider relatively old enclosures
                    xi, yi = enc_pos
                    
                    # Get environmental values
                    veg_value = self.model.veg_ras[to_numpy_y(self.model, yi), xi]
                    ag_value = self.model.ag_ras[to_numpy_y(self.model, yi), xi]
                    suit_val = self.model.suitability_raster[to_numpy_y(self.model, yi), xi]
                    
                    # Calculate base score
                    base_score = 10 - (age/2)
                    
                    # Apply environmental factors
                    env_factor = 1.0 + (veg_value * 0.02) + (ag_value * 0.02) + (suit_val * 0.02)
                    final_score = base_score * env_factor
                    
                    # Add randomization (±30%)
                    random_factor = random.uniform(0.7, 1.3)
                    bst_cells.append([enc_pos, final_score * random_factor])
        
        # Check if reusing is a viable strategy
        has_reuse_candidates = False
        if bst_cells:
            average_score = sum(cell[1] for cell in bst_cells) / len(bst_cells)
            if average_score > 8:  # Threshold for considering reuse as viable
                has_reuse_candidates = True
        
        if self.model.enclosures:
            # Consider new locations only if no good reuse candidates or by random chance
            should_consider_new = not has_reuse_candidates or random.random() > 0.7
        else:
            should_consider_new = random.random() > 0.4
        
        if should_consider_new:
            # Consider other cells for new enclosures
            for cell in nbr:
                # Skip cells that already have enclosures recently built by others
                if any(e[0] == cell and (self.model.year - e[1]) < 25 for e in self.model.enclosures):
                    continue
                    
                xi, yi = cell
                
                # Get comprehensive environmental assessment
                veg_value = self.model.veg_ras[to_numpy_y(self.model, yi), xi]
                ag_value = self.model.ag_ras[to_numpy_y(self.model, yi), xi]
                suit_val = self.model.suitability_raster[to_numpy_y(self.model, yi), xi]
                
                # Calculate combined environmental score
                env_score = (suit_val * 0.7) + (veg_value * 0.15) + (ag_value * 0.15)
                
                # Only consider locations with good environmental values
                if env_score > mean_val + 1:
                    # New locations get a lower base score
                    base_score = env_score * 0.8
                    
                    # Add randomization (±25%)
                    random_factor = random.uniform(0.75, 1.25)
                    bst_cells.append([cell, base_score * random_factor])
        
        # If no suitable cells were found, return the agent's current position
        if not bst_cells:
            return self.pos
        else:
            # Use weighted random selection
            weights = [max(0, cell[1]) for cell in bst_cells]  # Ensure weights are non-negative
            total_weight = sum(weights)

            if total_weight > 0:
                # Normalize weights
                normalized_weights = [w / total_weight for w in weights]
                selected_idx = random.choices(range(len(bst_cells)), weights=normalized_weights, k=1)[0]
                selected_cell = bst_cells[selected_idx][0]
            else:
                # Fallback if all weights are zero (or there are no cells)
                if bst_cells:
                    # Select randomly without weights if weights are all zero
                    selected_idx = random.randint(0, len(bst_cells) - 1)
                    selected_cell = bst_cells[selected_idx][0]
    
        return selected_cell
    def update_own_suitability_raster(self):
        if self.own_suitability_raster is None:
            # Create a copy of the suitability raster to avoid modifying the original
            self.own_suitability_raster = self.model.suitability_raster.copy()

        # Adjust raster based on the agent's territory
        if self.territory:
            # Create a binary mask of the territory
            territory_mask = np.zeros_like(self.own_suitability_raster, dtype=bool)
            for x, y in self.territory:
                numpy_y = to_numpy_y(self.model, y)
                territory_mask[numpy_y, x] = True
            
            # Calculate the distance transform (distance from territory boundaries)
            distances = distance_transform_edt(~territory_mask)
            
            # Apply a distance-based adjustment to the suitability raster
            adjustment = np.where(distances <= 10, 1 / (distances + 1), 0)
            self.own_suitability_raster += adjustment
        
        # Adjust raster based on the agent's memory of past locations
        if self.memory:
            for (x, y), value, year in self.memory:
                numpy_cell = (to_numpy_y(self.model, y), x)
                diff_y=self.model.year - year
                if diff_y==0:
                    self.own_suitability_raster[numpy_cell] -=0.5
                else:
                    time_factor = 1 / (diff_y)  # Weight decreases with time
                    if value >= 5:
                        # Increase suitability for positive memories
                        self.own_suitability_raster[numpy_cell] += value * time_factor * 0.2
                    elif value > 0:
                        # Decrease suitability for negative memories
                        self.own_suitability_raster[numpy_cell] -= (1 / value) * time_factor * 0.2


# Model

## Model funcs

**Core Functions for the Model**

This section defines the essential functions that drive the behavior agents and their interactions with the environment. These functions handle tasks such as calculating environmental suitability, placing households and members, evaluating neighborhoods, visualizing the model's state, and simulating environmental degradation.

---

**1. Utility and Environmental Functions**

- **`to_numpy_y`**:
  - Converts Mesa y-coordinates to NumPy y-coordinates (Mesa's y-axis is inverted).

- **`env_mean_val`**:
  - Calculates the mean environmental suitability value in a neighborhood.

- **`calculate_carrying_capacity`**:
  - Determines the maximum number of agents the environment can support based on suitability and population needs.

- **`eval_neighborhood`**:
  - Evaluates the suitability of a neighborhood for placing a household, considering factors like carrying capacity and existing agents.

- **`overlap_territory`**:
  - Checks if a neighborhood overlaps with another household's territory.

---

**2. Household Placement Functions**

- **`place_household`**:
  - Places a household agent in the environment based on suitability, territory, and memory.
  - Adjusts the suitability raster using:
    - **Territory**: Increases suitability near the agent's territory.
    - **Memory**: Incorporates past experiences (positive or negative) to influence placement.
  - Uses probability weights to select a valid position that avoids overlaps and satisfies neighborhood conditions.

- **`place_members`**:
  - Places household members within a specified territory.
  - Adjusts suitability based on the agent's memory and selects a position using probability weights.

---

**3. Environmental Degradation Function**

- **`env_degrade`**:
  - Simulates environmental degradation around a given position.
  - Degrades suitability values in a neighborhood based on:
    - **Distance-based degradation**: Degradation decreases with distance from the center.
    - **Position-based degradation**: Higher degradation at the center position.
  - Uses vectorized operations for efficient calculations.

---

**4. Visualization Function**

- **`viz_map`**:
  - Visualizes the model's state, including the suitability raster, household agents, household members, and enclosures.
  - Saves the visualization as an image file (`year_i_map.png`).

In [ ]:
# Convert Mesa y-coordinate to NumPy y-coordinate (Mesa's y-axis is inverted)
def to_numpy_y(model, mesa_y):
    numpy_y = np.clip(model.grid.height - mesa_y - 1, 0, model.grid.height - 1)
    return numpy_y

# Calculate the mean environmental suitability value in a neighborhood
def env_mean_val(model, current_pos, r):
    # Get neighborhood cells within radius r
    nbr = model.grid.get_neighborhood(
        current_pos, moore=True, include_center=False, radius=r
    )
    # Filter out-of-bounds cells and cells beyond x=280
    nbr = [cell for cell in nbr if not model.grid.out_of_bounds(cell) and cell[0] < 280]
    # Extract suitability values for each cell in the neighborhood
    env_values = [
        model.suitability_raster[to_numpy_y(model, pos[1]), np.clip(pos[0], 0, 279)]
        for pos in nbr
    ]
    # Calculate the mean suitability value
    mean_env_value = sum(env_values) / len(env_values)
    return mean_env_value

# Calculate the carrying capacity of the environment based on suitability and population needs
def calculate_carrying_capacity(model, suitability_raster, population, needs):
    """Calculates the carrying capacity based on average suitability."""
    max_agent_resources = population * needs  # Total resources needed by the population
    carrying_capacity = np.sum(suitability_raster) / max_agent_resources  # Total suitability divided by needs
    return int(np.floor(carrying_capacity))  # Round down to the nearest integer

# Evaluate the suitability of a neighborhood for placing a household
def eval_neighborhood(model, x, y, household, radius, n):
    # Get neighborhood cells within the given radius
    nbr = model.grid.get_neighborhood((x, y), moore=True, include_center=False, radius=radius)
    # Count non-empty cells in the neighborhood
    for cell in nbr:
        if model.grid.is_cell_empty(cell) is False:
            n += 1
    # Convert y-coordinate to NumPy format
    numpy_y = model.suitability_raster.shape[0] - y - 1
    # Define boundaries for the subarray (neighborhood area)
    start_x = max(0, x - radius)
    stop_x = min(model.suitability_raster.shape[1], x + radius + 1)
    start_numpy_y = max(0, numpy_y - radius)
    stop_numpy_y = min(model.suitability_raster.shape[0], numpy_y + radius + 1)
    # Extract subarrays for suitability and return rasters
    subarray = model.suitability_raster[start_numpy_y:stop_numpy_y, start_x:stop_x]
    ret_array = model.return_raster[start_numpy_y:stop_numpy_y, start_x:stop_x]
    # Adjust suitability values if return raster values are >= 2
    if np.any(ret_array >= 2):
        mask2 = ret_array >= 2
        subarray[mask2] += ret_array[mask2]
    # Calculate the carrying capacity for the neighborhood
    eval_env = calculate_carrying_capacity(model, subarray, household.manpower, n)
    return eval_env

# Check if a neighborhood overlaps with another household's territory
def overlap_territory(model, current_agent, nbr):
    l = []
    Overlap = 0
    # Iterate through all households except the current one
    for household in model.agents_by_type[Household_Agent]:
        if household != current_agent and household.territory != None:
            # Check if any cell in the neighborhood overlaps with the household's territory
            if any(cell in household.territory for cell in nbr):
                Overlap += 1
            else:
                Overlap += 0
            # Calculate the overlap ratio
            overlap_state = Overlap / (len(nbr))
            # Return True if overlap exceeds 25%
            if overlap_state > 0.25:
                return True
            else:
                return False

# Place a household agent in the environment based on suitability and memory
def place_household(model, current_agent, suitability_raster):
    # Create a copy of the suitability raster to avoid modifying the original
    current_agent.update_own_suitability_raster()       
    # Generate probability weights for placement
    current_agent.own_suitability_raster[current_agent.own_suitability_raster < 0]=0
    prob_weights = current_agent.own_suitability_raster.flatten() ** 3  # Weight by suitability cubed
    prob_weights *= model.place_raster.flatten()  # Set forbidden areas to 0

    # Normalize probabilities
    total_weight = np.sum(prob_weights)
    if total_weight != 0:
        prob = prob_weights / total_weight
    else:
        prob = np.zeros_like(prob_weights)
    prob[np.isnan(prob)] = 0  # Handle NaN values
    
    # Function to find a valid position based on probabilities
    def get_valid_position():
        while True:
            # Randomly select a position based on probabilities
            random_index = np.random.choice(np.arange(len(prob)), p=prob)
            y, x = np.unravel_index(random_index, suitability_raster.shape)
            mesa_y = suitability_raster.shape[0] - y - 1
            # Check if the position and its neighborhood are within bounds
            if (20 <= x < model.grid.width - 20 and 
                20 <= mesa_y < model.grid.height - 20 and
                20 <= y < suitability_raster.shape[0] - 20 and
                20 <= x < suitability_raster.shape[1] - 20 and
                model.place_raster[y, x] == 1):
                return x, mesa_y
    
    # Get initial position
    x, y = get_valid_position()
    nbr = model.grid.get_neighborhood((x, y), moore=True, include_center=False, radius=20)
    
    # Keep trying new positions until one satisfies all conditions
    while (overlap_territory(model, current_agent, nbr) and 
           eval_neighborhood(model, x, y, current_agent, 20, 35) < 1):
        x, y = get_valid_position()
        nbr = model.grid.get_neighborhood((x, y), moore=True, include_center=False, radius=20)
    
    return (x, y)

# Place household members within a specified territory
def place_members(model, current_agent,  ter):
    # Create a copy of the suitability raster and mask forbidden areas
    current_agent.update_own_suitability_raster()
    own_suitability_raster=current_agent.own_suitability_raster
    # Filter territory cells to include only valid positions
    ter = [cell for cell in ter if model.place_raster[to_numpy_y(model, cell[1]), cell[0]]]
    # Extract suitability values for the territory
    suitability_values = [
        own_suitability_raster[to_numpy_y(model, cell[1]), cell[0]] for cell in ter
    ]
    
    # Generate probability weights (cubed to emphasize high suitability)
    prob_weights = np.array(suitability_values) ** 3
    total_weight = np.sum(prob_weights)

    # Normalize probabilities
    if total_weight > 0:
        prob = prob_weights / total_weight
    else:
        prob = np.zeros_like(prob_weights)
    
    # Select a cell within the territory based on probabilities
    selected_index = np.random.choice(len(ter), p=prob)
    selected_cell = ter[selected_index]
    return selected_cell  # Return the selected cell coordinates

# Simulate environmental degradation around a position
def env_degrade(model, mesa_x, mesa_y, r, d_factor, p_factor):
    """
    Optimized version of environmental degradation calculation.
    Uses vectorized operations where possible and pre-filters invalid cells.
    
    Args:
        model: Model instance containing the grid and suitability raster
        mesa_x, mesa_y: Position coordinates in Mesa system
        r: Radius for neighborhood calculation
        d_factor: Distance-based degradation factor
        p_factor: Position-based degradation factor
    """
    # Convert Mesa y-coordinate to NumPy y-coordinate
    numpy_y = model.grid.height - mesa_y - 1
    
    # Get neighborhood cells within the given radius
    nbr = model.grid.get_neighborhood((mesa_x, mesa_y), moore=True, include_center=False, radius=r)
    
    # Convert neighborhood cells to NumPy arrays for vectorized operations
    cells = np.array(nbr)
    x_coords = cells[:, 0]
    y_coords = model.grid.height - cells[:, 1] - 1  # Transform all y-coordinates at once
    
    # Create a mask for valid coordinates
    valid_mask = (
        (x_coords >= 0) & 
        (x_coords < model.suitability_raster.shape[1]) &
        (y_coords >= 0) & 
        (y_coords < model.suitability_raster.shape[0])
    )
    
    # Filter valid coordinates
    valid_x = x_coords[valid_mask]
    valid_y = y_coords[valid_mask]
    original_y = cells[valid_mask][:, 1]  # Original Mesa y-coordinates for distance calculation
    
    # Calculate distances vectorized
    x_distances = np.abs(mesa_x - valid_x)
    y_distances = np.abs(mesa_y - original_y)
    distance_factors = d_factor / ((x_distances + y_distances) / 2)
    
    # Update suitability values vectorized
    current_values = model.suitability_raster[valid_y, valid_x]
    new_values = np.maximum(current_values - distance_factors, 0.0001)  # Ensure values don't drop below 0.0001
    model.suitability_raster[valid_y, valid_x] = new_values
    
    # Update the center position with a higher degradation factor
    model.suitability_raster[numpy_y, mesa_x] = max(
        model.suitability_raster[numpy_y, mesa_x] - p_factor,
        0.0001
    )

In [ ]:
def viz_map(model, suitability_raster, year, run_dir):
    """
    Visualize the model's state, including suitability raster and agent positions.
    Saves the figure without displaying it during runtime.
    
    Args:
        model: The simulation model instance.
        suitability_raster: The environmental suitability raster.
        year: Current simulation year.
        run_dir: Directory to save visualization.
    """
    # Set up visualization theme and figure
    sns.set_theme(style="whitegrid")
    height, width = suitability_raster.shape
    
    # Create figure with the plt.ioff() to prevent display
    plt.ioff()  # Turn off interactive mode
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot the suitability raster with fixed scale 0-10
    im = ax.imshow(suitability_raster, cmap="YlGn", origin="upper", vmin=0, vmax=10)
    
    # Add colorbar for the scale
    cbar = fig.colorbar(im, ax=ax, orientation='vertical', shrink=0.8)
    cbar.set_label('Suitability Index (0-10)')
    
    # Assign unique colors to households
    household_dict = {}
    color_palette = sns.color_palette("tab20", n_colors=len(model.agents_by_type[Household_Agent]))
    color_index = 0
    
    # Plot household agents
    for agent in model.agents_by_type[Household_Agent]:
        numpy_row = height - 1 - agent.pos[1]
        numpy_column = agent.pos[0]
        agent_color = color_palette[color_index]
        sns.scatterplot(x=[numpy_column], y=[numpy_row], color=agent_color, s=20, marker="o", ax=ax)
        household_dict[agent] = agent_color
        color_index += 1
    
    # Plot household members
    for agent in model.agents_by_type[Household_Agent]:
        for c in agent.memory:
            if c[-1] == year:
                numpy_row = height - 1 - c[0][1]
                numpy_column = c[0][0]
                a_color = household_dict[agent]
                sns.scatterplot(x=[numpy_column], y=[numpy_row], color=a_color, s=10, marker="o", ax=ax)
    
    # Plot enclosures
    for enc in model.enclosures:
        numpy_row = height - 1 - enc[0][1]
        numpy_column = enc[0][0]
        #Check if the agent exists in household_dict
        if enc[2] in household_dict:
            enc_color = household_dict[enc[2]]
        else:
            # Use a default color if agent not found
            enc_color = 'gray'
        if year > 0:
            opc = (1 - (year - enc[1]) / year)
        else:
            opc = 1
        sns.scatterplot(x=[numpy_column], y=[numpy_row], color=enc_color, s=15, marker="s", ax=ax, alpha=opc)
    
    # Format title and axes
    plt.title(f"Year {year} Simulated Map", fontsize=14)
    plt.xticks([])
    plt.yticks([])
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Save the visualization
    filename = f'year_{year}_map.png'
    filepath = os.path.join(run_dir, filename)
    plt.savefig(filepath, dpi=400)
    
    # Close the figure to free memory
    plt.close(fig)

## model_initialization

### **Weights Determination and Adjustment**



This section handles the determination, adjustment, and visualization of weights for various environmental and human factors in the model. These weights influence decision-making processes, such as encampment site selection.

---

#### **1. Weight Determination**

- **`detrmine_weights`**:
  - Randomly assigns weights to key factors (rain, slope, others) while ensuring no single factor dominates.
  - Distributes weights across specific factors (e.g., agricultural suitability, distance to water, etc.).
  - Stores weights in dictionaries (`permanent_weights_dict` and `yearly_weights_dict`) for permanent and yearly factors.

---

#### **2. Interactive Weight Adjustment**

- **`create_sliders_and_weights`**:
  - Creates interactive sliders using `ipywidgets` to adjust the importance of each factor.
  - Updates weights dynamically based on slider values.
  - Normalizes weights to ensure their sum equals 1.

---

#### **3. Loading Previous Weights**

- Loads weights from the most recent run folder using JSON files (`permanent_weights_dict.json` and `yearly_weights_dict.json`).
- Ensures continuity between simulation runs by reusing previously defined weights.

---

#### **4. Visualizing Weights**

- **`plot_weights`**:
  - Combines permanent and yearly weights into a single dictionary.
  - Plots weights as a bar chart for easy visualization.
  - Displays weight values on top of each bar for clarity.

In [ ]:
# Function to randomly determine weights for environmental factors
def detrmine_weights():
    # Randomly assign weights for rain, slope, and other factors
    rain = random.uniform(0.1, 0.95)  
    slope = random.uniform(0.1, 0.95)  
    others = random.uniform(0.1, 0.95) 

    # Calculate the total weight
    total = rain + slope + others
    # Ensure weights are balanced and no single factor dominates
    while total > 1 or others > rain or others > slope:
        rain = random.uniform(0.1, 0.95)  
        slope = random.uniform(0.1, 0.95)  
        others = random.uniform(0.1, 0.95)
        total = rain + slope + others

    # Calculate the weight for non-specified factors
    non = 1 - rain - slope - others  
    values = [rain, slope, others, non]  # Store weights in a list
    
    return values

# Generate weights and store them in `values`
values = detrmine_weights()

# Distribute weights across specific factors
v = [values[0] / 2] * 2  # Split rain weight into two equal parts
v1 = [values[1] / 2] * 2  # Split slope weight into two equal parts
v2 = [values[2] / 3] * 3  # Split "others" weight into three equal parts
nv = [values[3] / 7] * 7  # Split "non" weight into seven equal parts

# Assign weights to specific factors
wp6, ws6 = v  # Rain-related weights
wp7, wp8 = v1  # Slope-related weights
wp1, wp2, wp3, wp5, ws3, ws4, ws5 = nv  # Non-specified weights
ws1, ws2, wp4 = v2  # Return, stress, and permanent water weights

# Combine all weights into a single list
wts = [wp1, wp2, wp3, wp4, wp5, wp6, wp7, wp8, ws1, ws2, ws3, ws4, ws5, ws6]

# Print the generated weights and their sum
print(values, sum(values))

# Create dictionaries to store weights for permanent and yearly factors
permanent_weights_dict = {
    #"p_agri": wp1,  # Agricultural suitability
    "dist_to_kb": wp2,  # Distance to Kadesh Barnea
    #"aspect": wp3,  # Terrain aspect
    "p_water": wp4,  # Permanent water sources
    #"p_veg_fit": wp5,  # Vegetation suitability
    "Mean_rain": wp6,  # Mean annual rainfall
    #"slope_range": wp7,  # Slope range
    "slope_suitability": wp8,  # Slope suitability
}

yearly_weights_dict = {
    "return_to_site": ws1,  # Return to previous sites
    "humen_stress": ws2,  # Human stress
    #"pasture_y": ws3,  # Pasture yield
    #"yearly_agri": ws4,  # Yearly agricultural suitability
    #"yearly_water": ws5,  # Yearly water availability
    "Yearly_rain": ws6,  # Yearly rainfall
}

In [ ]:
import ipywidgets as widgets
from ipywidgets import interactive
from IPython.display import display

# Function to create interactive sliders for weight adjustment
def create_sliders_and_weights():
    # Combine permanent and yearly weights into a single dictionary
    all_weights = {**permanent_weights_dict, **yearly_weights_dict}
    
    # Create sliders for each weight
    sliders = {
        key: widgets.IntSlider(min=0, max=7, step=1, value=1, description=key, layout=widgets.Layout(width='400px'))
        for key in all_weights.keys()
    }
    
    # Function to update weights when sliders are adjusted
    def update_weights(change):
        total = sum(slider.value for slider in sliders.values())
        for key, slider in sliders.items():
            weight = slider.value / total  # Normalize weights
            if key in permanent_weights_dict:
                permanent_weights_dict[key] = weight  # Update permanent weights
            else:
                yearly_weights_dict[key] = weight  # Update yearly weights
    
    # Attach the update function to each slider
    for slider in sliders.values():
        slider.observe(update_weights, names='value')
    
    # Display the sliders in a vertical layout
    display(widgets.VBox([widgets.Label("Adjust Importance")] + list(sliders.values())))   

# Call the function to create and display sliders
create_sliders_and_weights()

In [ ]:
# Find the most recent run folder
run_folder = max([f.path for f in os.scandir(r'C:\Users\Owner\Documents\Itay\thesis\ABM\runs_simp') if f.is_dir()])

# Load permanent weights from the previous run
with open(os.path.join(run_folder, 'permanent_weights_dict.json')) as f:
    permanent_weights_dict = json.load(f)

# Load yearly weights from the previous run
with open(os.path.join(run_folder, 'yearly_weights_dict.json')) as f:
    yearly_weights_dict = json.load(f)

In [ ]:
# Combine permanent and yearly weights into a single dictionary
weights = {**permanent_weights_dict, **yearly_weights_dict}
wts = list(weights.values())  # Extract weight values

# Function to plot weights as a bar chart
def plot_weights(weights):
    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(weights.keys())  # Factor names
    values = list(weights.values())  # Weight values
    
    # Create bar chart
    bars = ax.bar(names, values)
    ax.set_ylabel('Weight')
    ax.set_title('Weights for Different Factors')
    plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for readability
    
    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height,
                f'{height:.3f}',
                ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# Plot the weights
plot_weights(weights)

In [ ]:
weights

### **Suitability Raster Generation**



This section generates a suitability raster for a given year based on environmental factors, weights, and spatial data. The suitability raster is used to evaluate the desirability of different locations for encampment placement.

---

#### **Key Steps**:

1. **Select Yearly Data**:
   - The function selects raster data for the current year from the `y_output` dataset using the provided `indices`.

2. **Unpack Weights**:
   - Weights for both permanent and yearly factors are unpacked and used to calculate their contributions to the suitability raster.

3. **Summarize Permanent Factors**:
   - Permanent factors (e.g., agricultural suitability, slope) are summarized using their respective weights.

4. **Calculate Yearly Parameters**:
   - Additional yearly parameters (e.g., human stress, return to site) are calculated using the `Yi_params` function.

5. **Summarize Yearly Factors**:
   - Yearly factors (e.g., pasture yield, yearly rainfall) are summarized using their respective weights.

6. **Resample and Mask Raster**:
   - The suitability raster is resampled to a 250m cell size using bilinear interpolation.
   - The raster is then masked using a polygon to focus on the study area.

7. **Convert to NumPy Array**:
   - The masked raster is converted to a NumPy array for use in the model.

8. **Placement Raster**:
   - A separate raster defining the placement extent is loaded and converted to a NumPy array for use in the model.

---

#### **Outputs**:
- **`suitability_raster`**: A NumPy array representing the suitability of each location for encampment placement.
- **`past_years`**: Data related to past years, used for historical analysis.
- **`yi_hu_stressed_z`**: Human stress data for the current year, used to evaluate environmental impact on agents.

In [ ]:
# Load the placement extent raster
place_ras = Raster(r'C:\Users\Owner\Documents\Itay\Prj_25.gdb\placement_extent_Raster')
pext = Raster(param0)
lowerLeft = arcpy.Point(pext.extent.XMin, pext.extent.YMin)
polygon_mask = r"C:\Users\Owner\Documents\Itay\thesis\HN_ext.shp"
resampled_raster = (r"C:\Users\Owner\Documents\Itay\resampled_slp1.tif")
masked_raster = arcpy.sa.ExtractByMask(resampled_raster, polygon_mask)
resampled_raster_obj = arcpy.Raster(masked_raster)
slpras=arcpy.RasterToNumPyArray(resampled_raster_obj, lower_left_corner=lowerLeft, 
                                        ncols=new_cols, nrows=new_rows, nodata_to_value=0)
# Convert the placement raster to a NumPy array
place_raster = arcpy.RasterToNumPyArray(place_ras, lower_left_corner=lowerLeft, 
                                        ncols=new_cols, nrows=new_rows, nodata_to_value=0)

In [ ]:
def get_suitability_raster(y_output, indices, year, yearly_outputs, weights,slope_array=slpras):
    """
    Generates a suitability raster for a given year based on factors and weights.
    
    Args:
        y_output: Output data for yearly factors.
        indices: Indices to access specific years in `y_output`.
        year: The current year of the simulation.
        all_outputs: All output data for the simulation.
        weights: List of weights for environmental factors.
    
    Returns:
        suitability_raster: A NumPy array representing the suitability raster.
        past_years: Data related to past years.
        yi_hu_stressed_z: Human stress data for the current year.
    """
    # Select the raster data for the current year
    random_year = y_output[indices[year]]
    
    # Unpack weights for permanent and yearly factors
    wp2,wp4, wp6,wp8, ws1, ws2, ws6 = weights
    
    # Summarize permanent factors using their weights
    ws_p = summarize_outputs_p( wp2,  wp4, wp6, wp8)    
    
    # Unpack yearly factors from the selected year's data
    calc_pastoral_Yi, agr_raster, twFuzzyMember_calc, YrainFuzzyMember_calc = random_year
    
    # Calculate additional yearly parameters (e.g., return raster, human stress)
    return_ras, yi_hu_stressed_z, past_years = Yi_params(param10, year, yearly_outputs)
    
    # Summarize yearly factors using their weights
    res = summarize_outputs_y(ws1, ws2, ws6, ws_p, return_ras, yi_hu_stressed_z,
                               YrainFuzzyMember_calc, year)
    
    # Define paths for resampling and masking
    resampled_raster = (r"C:\Users\Owner\Documents\Itay\resampled_raster.tif")
    polygon_mask = r"C:\Users\Owner\Documents\Itay\thesis\HN_ext.shp"
    
    # Resample the raster to the desired cell size (250m) using bilinear interpolation
    arcpy.management.Resample(in_raster=res, 
                              out_raster=resampled_raster, 
                              cell_size=250, 
                              resampling_type="Bilinear")  # Bilinear Resampling
    
    # Mask the raster using the specified polygon
    masked_raster = arcpy.sa.ExtractByMask(resampled_raster, polygon_mask)
    resampled_raster_obj = arcpy.Raster(masked_raster)
    
    # Get the extent of the placement raster
    pext = Raster(param0)
    lowerLeft = arcpy.Point(pext.extent.XMin, pext.extent.YMin)
    
    # Convert the masked raster to a NumPy array
    suitability_raster = arcpy.RasterToNumPyArray(resampled_raster_obj, lower_left_corner=lowerLeft, 
                                                  ncols=new_cols, nrows=new_rows, nodata_to_value=0)
    slp_lim = np.where(slope_array<1, 0, 1)
    suitability_raster=suitability_raster*slp_lim
    
    return suitability_raster, past_years, yi_hu_stressed_z



### **NomadModel Class**



The `NomadModel` class is the core of the simulation, managing agents, the environment, and the progression of time. It handles agent placement, environmental updates, and data collection.

---

#### **Key Features**:

1. **Initialization**:
   - Sets up the grid, rasters, and agents.
   - Assigns territories and initial positions to households and their members.
   - Simulates environmental degradation based on agent placement.

2. **Agent Creation**:
   - Each household consists of 10 members, including a leader with additional resources.
   - Households and members are placed in valid locations based on suitability rasters.

3. **Step Function**:
   - Advances the simulation by one year.
   - Calls `step_start`, `step`, and `step_end` methods for household members and agents.
   - Stores yearly outputs and collects data for analysis.

4. **Yearly Movement**:
   - Updates rasters (suitability, return, stress) for the new year.
   - Resets agent positions and initializes the new year for households and members.

5. **Data Collection**:
   - Uses `DataCollector` to track agent attributes (e.g., manpower, surplus, position) for analysis.

---

#### **Methods**:

- **`__init__`**:
  - Initializes the model with rasters, weights, and agents.
  - Places households and members in valid locations.

- **`step`**:
  - Advances the simulation by one year.
  - Updates agent states and collects data.

- **`move_year`**:
  - Moves the simulation to the next year by updating rasters and resetting positions.

- **`reset_pos`**:
  - Resets the positions of all agents at the start of a new year.

In [ ]:
class NomadModel(mesa.Model):
    """
    The main model class of the simulation.
    Manages agents, the environment, and the simulation steps.
    """
    def __init__(self, suitability_raster, return_raster, stress_ras, inds, place_raster=place_raster, weights=wts):
        """
        Initialize the model with rasters, weights, and agents.
        
        Args:
            suitability_raster: Raster representing environmental suitability.
            return_raster: Raster representing return-to-site suitability.
            stress_ras: Raster representing human stress.
            inds: Indices for accessing yearly data.
            place_raster: Raster defining valid placement areas.
            weights: List of weights for environmental factors.
        """
        super().__init__()
        self.num_agents = Num_agents()  # Number of agents in the model
        height, width = suitability_raster.shape  # Dimensions of the raster
        self.grid = mesa.space.MultiGrid(width, height, False)  # MultiGrid for agent placement
        self.yearly_outputs = []  # Store yearly outputs for analysis
        self.suitability_raster = suitability_raster  # Environmental suitability raster
        self.target_raster = np.zeros_like(suitability_raster)  # Target raster for calculations
        self.return_raster = return_raster  # Return-to-site suitability raster
        self.place_raster = place_raster  # Raster defining valid placement areas
        self.ras_control = []  # Store copies of suitability raster for debugging
        self.weights = weights  # Weights for environmental factors
        self.year = 0  # Current simulation year
        self.territories = []  # List of territories claimed by households
        self.enclosures = []  # List of enclosures built by households
        self.suitability_raster[np.isnan(self.suitability_raster)] = 0  # Replace NaN values with 0
        self.threshold = 2  # Surplus Threshold for decision-making
        self.scheduled_new_agents = []  # Initialize the list for scheduled agent creations

        # Convert rasters to NumPy array
        lowerLeft = arcpy.Point(stress_ras.extent.XMin, Raster(param10).extent.YMin)
        self.veg_ras = arcpy.RasterToNumPyArray(resample(y_output[inds[0]][0], os.path.join(dump_directory, "resamp1")),
                                 lower_left_corner=lowerLeft, ncols=new_cols,
                                 nrows=new_rows, nodata_to_value=0)
        self.ag_ras = arcpy.RasterToNumPyArray(resample(y_output[inds[0]][1], os.path.join(dump_directory, "resamp3")),
                                 lower_left_corner=lowerLeft, ncols=new_cols,
                                 nrows=new_rows, nodata_to_value=0)
        self.stress_ras = arcpy.RasterToNumPyArray(resample(stress_ras, os.path.join(dump_directory, "resamp2")),
                                              lower_left_corner=lowerLeft, ncols=new_cols,
                                              nrows=new_rows, nodata_to_value=0)
        
        self.ras_control.append(self.suitability_raster.copy())  # Store initial suitability raster

        # Create and place agents
        for i in range(self.num_agents):
                        
            # Create the household agent
            agent = Household_Agent(self, threshold=self.threshold)
            mesa_x, mesa_y = place_household(self, agent, self.suitability_raster)  # Place household
            self.grid.place_agent(agent, (mesa_x, mesa_y))
            self.target_raster[to_numpy_y(self, agent.pos[1]), agent.pos[0]] += 0.75
            territory = self.grid.get_neighborhood((mesa_x, mesa_y), moore=True, include_center=True, radius=25)
            agent.territory = territory  # Assign territory to the household
            self.territories.append(agent.territory)
            
            for _ in range (9):
                selected_cell = place_members(self, agent,  agent.territory)  # Place member
                env_degrade(self, selected_cell[0], selected_cell[1], 11, 0.3, 0.5)  # Simulate member degradation
                agent.memory.append([selected_cell, env_mean_val(self, selected_cell, 5), self.year])
                self.target_raster[to_numpy_y(self, selected_cell[1]), selected_cell[0]] += 0.5
            env_degrade(self, mesa_x, mesa_y, 25, 0.5, 0.9)  # Simulate household level environmental degradation
            agent.memory.append([agent.pos, env_mean_val(self, agent.pos, 20), self.year])  # Store memory

            

        # Set up DataCollector for agent data
        self.datacollector = DataCollector(agent_reporters={"Manpower": "manpower","flocks":"flock_head",
                                            "surplus": "surplus","position": "pos","enclosures": "enc_memory"})
        self.datacollector.collect(self)  # Collect initial data

    def step(self):
        """
        Advance the model by one step (year).
        """
        #self.agents_by_type[household_member].do("step_start")  # Start step for household members
        self.agents_by_type[Household_Agent].shuffle_do("step")  # Step for household agents
        #self.agents_by_type[household_member].do("step_end")  # End step for household members
        self.yearly_outputs.append(self.target_raster.copy())  # Store yearly output
        self.year += 1  # Increment year
        self.datacollector.collect(self)  # Collect data for the current step

    def move_year(self, inds):
        """
        Move the model to the next year by updating rasters and resetting positions.
        """
        self.reset_pos()  # Reset agent positions
        # Update suitability, return, and stress rasters for the new year
        self.suitability_raster, self.return_raster, stress_ras = get_suitability_raster(y_output, inds, self.year, self.yearly_outputs, self.weights)
        self.ras_control.append(self.suitability_raster.copy())  # Store updated suitability raster
        lowerLeft = arcpy.Point(stress_ras.extent.XMin, stress_ras.extent.YMin)
        self.veg_ras = arcpy.RasterToNumPyArray(resample(y_output[inds[self.year]][0], os.path.join(dump_directory, "resamp1")),
                                     lower_left_corner=lowerLeft, ncols=new_cols,
                                     nrows=new_rows, nodata_to_value=0)
        self.ag_ras = arcpy.RasterToNumPyArray(resample(y_output[inds[self.year]][1], os.path.join(dump_directory, "resamp3")),
                                     lower_left_corner=lowerLeft, ncols=new_cols,
                                     nrows=new_rows, nodata_to_value=0)
        self.stress_ras = arcpy.RasterToNumPyArray(resample(stress_ras, os.path.join(dump_directory, "resamp2")),
                                                  lower_left_corner=lowerLeft, ncols=new_cols,
                                                  nrows=new_rows, nodata_to_value=0)
        self.target_raster = np.zeros_like(self.suitability_raster)  # Reset target raster
        if hasattr(self, 'scheduled_new_agents') and self.scheduled_new_agents:
            for resources in self.scheduled_new_agents:
                self.create_replacement_agent(resources)
            self.scheduled_new_agents = []  # Reset the scheduled agent creations
        self.agents_by_type[Household_Agent].shuffle_do("year_initiation")  # Initialize new year for households
        #self.agents_by_type[household_member].shuffle_do("year_initiation")  # Initialize new year for members

    def reset_pos(self):
        """
        Reset positions of all agents.
        """
        for agent in self.agents:
            self.grid.remove_agent(agent)  # Remove agents from the grid
    def create_replacement_agent(self, resources):
        """
        Creates a new agent with resources from a removed agent plus additional flocks.
        Uses Mesa 3.0's create_agents method.

        Parameters:
        ----------
        resources : dict
            Dictionary containing 'manpower', 'surplus', and 'flock_head' values from the removed agent
        """
        # Calculate additional flocks to add
        additional_flocks = sum(np.floor(random.gauss(90, 15)) for _ in range(10))

        # Create a new agent using Mesa's create_agents method
        new_agents = Household_Agent.create_agents(
            model=self,
            n=1,
            threshold=self.threshold,
            manpower=resources['manpower'],
            surplus=resources['surplus'],
            flock_head=resources['flock_head'] + additional_flocks
        )

        # Get the newly created agent
        agent = new_agents[0]
        print(f"Created new agent {agent.unique_id} with manpower={agent.manpower}, flocks={agent.flock_head}, surplus={agent.surplus}")

## **Model Execution**



This section handles the execution of the simulation, including initialization, running the model for a specified number of years, and saving results.

---

#### **Key Steps**:

1. **Initialization**:
   - A timestamp is generated to create a unique directory for storing results.
   - Yearly data indices are shuffled to randomize environmental conditions.
   - Suitability, return, and stress rasters are generated for the initial year.
   - The `NomadModel` is initialized with the generated rasters.

2. **Simulation Loop**:
   - The model runs for the specified number of years (`model_years`).
   - Each year, the model advances one step using `model.step()`.
   - The current state of the model is visualized using `viz_map()` and saved to the run directory.
   - The model moves to the next year using `model.move_year()`.

3. **Data Collection and Saving**:
   - Data for household members and households is collected using the `DataCollector`.
   - The data is saved to CSV files (`members_data.csv` and `household_data.csv`).
   - Weights for permanent and yearly factors are saved to JSON files.

4. **Return the Model**:
   - The completed model instance is returned for further analysis or visualization.

---

#### **Usage**:
- To run the simulation, call `run_model()` with the desired number of years and weights.
- Example: `model = run_model(5, wts)` runs the simulation for 5 years.

In [ ]:
def run_model(model_years, wts, y_output=y_output):
    """
    Runs the simulation for a specified number of years.

    Args:
        model_years: Number of years to simulate.
        wts: List of weights for environmental factors.
        y_output: Yearly output data for environmental factors.

    Returns:
        model: The completed model instance.
    """
    # Turn off interactive plotting globally at the beginning
    plt.ioff()

    # Generate a timestamp for the run directory
    current_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    # Create a list of indices and shuffle them for random yearly data selection
    inds = list(range(len(y_output)))
    random.shuffle(inds)

    # Generate suitability, return, and stress rasters for the initial year
    suitability_raster, return_raster, stress_ras = get_suitability_raster(y_output, inds, year=0, yearly_outputs=[], weights=wts)

    # Initialize the model with the generated rasters
    model = NomadModel(suitability_raster, return_raster, stress_ras, inds)

    # Create a directory for storing run results
    run_directory = r"C:\Users\Owner\Documents\Itay\thesis\ABM\runs_simp\run_{}".format(current_date)
    if not os.path.exists(run_directory):
        os.makedirs(run_directory)

    # Run the simulation for the specified number of years with progress bar
    for i in tqdm(range(model_years), desc="Simulating years"):
        model.step()  # Advance the model by one year
        if i != (model_years - 1):
            viz_map(model, model.suitability_raster, i, run_directory)
            model.move_year(inds)
        else:
            viz_map(model, model.suitability_raster, i, run_directory)

    # Retrieve collected data from the model
    household_data = model.datacollector.get_agent_vars_dataframe()

    # Save data to CSV files
    household_data.to_csv(os.path.join(run_directory, "household_data.csv"))

    # Save weights to JSON files
    with open(os.path.join(run_directory, "permanent_weights_dict.json"), "w") as f:
        json.dump(permanent_weights_dict, f)
    with open(os.path.join(run_directory, "yearly_weights_dict.json"), "w") as f:
        json.dump(yearly_weights_dict, f)

    # Turn interactive mode back on at the end if needed for other visualizations
    plt.ion()

    return model

In [ ]:
# Run the model for x years with the given weights
#see=False
model = run_model(75, wts)
#if see:
#    create_plotly_dashboard(model)


## model_viz

### vizs

In [ ]:
yo_sum= sum(model.yearly_outputs)
#yo_sum[yo_sum < 1] = 0
#yo_sum[(yo_sum > 1) & (yo_sum < 2)] = 1
#yo_sum[yo_sum > 2] = 2

# Plot the raster using Matplotlib  
plt.imshow(yo_sum, cmap="viridis", origin="upper", vmax=2, vmin=0.75)
plt.colorbar()
plt.show()



In [ ]:
# Export the yo_sum numpy array to a raster in Arcpy
out_raster = os.path.join(dump_directory, "yo_sum")
arcpy.NumPyArrayToRaster(yo_sum, lower_left_corner=lowerLeft, x_cell_size=250, y_cell_size=250,
                         value_to_nodata=0).save(out_raster)


In [ ]:
diff_l=[]
diff_m=[]
meanarr=np.zeros_like(model.ras_control[0])
for i in range(len(model.ras_control)):
    meanarr+=model.ras_control[i]
    if i>0:
        suit_diff=model.ras_control[i-1]-model.ras_control[i]
        diff_l.append(suit_diff)
meanarr/=len(model.ras_control)
for i in range(len(model.ras_control)):
    diff_m.append(model.ras_control[i]-meanarr)

In [ ]:
diff_mm=[]
for i in diff_m:
    score=np.mean(i)
    diff_mm.append(score)

In [ ]:
plt.imshow(meanarr, cmap="YlGn", origin="upper",vmax=10, vmin=0)
plt.colorbar()
plt.show()

In [ ]:
y=0
for i in diff_m:
    print(y)
    plt.imshow(i, cmap="YlGn", origin="upper",vmin=-1.5, vmax=1.5)
    plt.colorbar()
    plt.show()
    y+=1

In [ ]:
y=1
for i in range(50,60):
    print(y)
    plt.imshow(i, cmap="YlGn", origin="upper",vmin=-1.5, vmax=1.5)
    plt.colorbar()
    plt.show()
    y+=1

#### data check

In [ ]:
def plot_surplus(data):
    mean_data=data.groupby("Step").mean()
    # Set up the figure size and style with Seaborn
    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid")
    sns.lineplot(x=mean_data.index, y=mean_data["surplus"], label="Mean surplus", color="r")
    # Add labels, title, and legend
    plt.xlabel("Step")
    plt.ylabel("Mean Value")
    #plt.title("Mean Manpower and Energy Over Time")
    plt.legend()

    # Show the plot
    plt.show()

# Get the data for all household members
def plot_all_surplus(data):
    # Get the data for all household members

    # Create the plot
    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")

    # Plot individual lines for each agent's surplus
    sns.lineplot(data=data, x="Step", y="surplus", alpha=0.3, color='gray', units="AgentID", estimator=None)

    # Add a line for the mean surplus
    sns.lineplot(data=data, x="Step", y="surplus", color='red', label='Mean Surplus', linewidth=2)

    plt.title("Changes in Household Members' Surplus Over Time")
    plt.xlabel("Step")
    plt.ylabel("Surplus")
    plt.show()

In [ ]:

household_data=model.datacollector.get_agent_vars_dataframe()
plot_all_surplus(household_data)
#plot_all_surplus(members_data)

# Calibration

## To gdf

In [ ]:
def to_gdf(lower_left_x,lower_left_y,cell_size,model_output, plot=False):
    yo_sum=sum(model_output.yearly_outputs)
    yo_sum[yo_sum < 1] = 0
    yo_sum[(yo_sum > 1.5)]= 1
    # Alternative approach if model.enclosures contains the enclosure positions directly
    for enclosure in model_output.enclosures:
        # Get the enclosure position in Mesa coordinates
        mesa_x, mesa_y = enclosure[0]
        
        # Convert from Mesa y (Cartesian) to numpy y 
        numpy_y = to_numpy_y(model_output,mesa_y)
        
        # Set the value to 2 for enclosure locations
        yo_sum[numpy_y, mesa_x] = 2
    #yo_sum[yo_sum > 2] = 2
    # Step 1: Find indices of cells with values > 0
    indices = np.column_stack((np.argwhere(yo_sum > 0), yo_sum[yo_sum > 0]))  # Shape: (N, 2), where N is the number of cells with value > 0

    # Step 2: Convert array indices to real-world coordinates
    real_world_coords = []
    for (row, col, value) in indices:
        # Calculate real-world coordinates
        x = lower_left_x + col * cell_size
        y = lower_left_y + (yo_sum.shape[0] - 1 - row) * cell_size
        real_world_coords.append((x, y, value))

    # Convert to a numpy array for easier manipulation
    real_world_coords = np.array(real_world_coords)
    # Step 3: Create a GeoDataFrame
    # Convert coordinates to Shapely Point objects
    geometry = [Point(x, y) for x, y, _ in real_world_coords]

    # Create a GeoDataFrame with the geometry and values
    gdf = gpd.GeoDataFrame({
        'geometry': geometry,
        'value': real_world_coords[:, 2]  # Include the values
    }, crs="EPSG:2039")
    if plot:
        # Plot using GeoPandas
        gdf.plot(column='value', cmap='viridis', legend=True, markersize=40, figsize=(7, 7))
        plt.title("Land Use in the Study Area")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.show()
    return gdf

In [ ]:
#define coords and co
pext=Raster(param0)
lowerLeft = arcpy.Point(pext.extent.XMin,pext.extent.YMin)
lower_left_x=lowerLeft.X
lower_left_y=lowerLeft.Y
# Cell size (in meters)
cell_size = 250
model_output=model
gdf_model=to_gdf(lower_left_x,lower_left_y,cell_size,model_output)
gdf_real=gpd.read_file(r'C:\Users\Owner\Documents\Itay\thesis\ABM\points_all\all_P.shp')

## optimize

In [ ]:
def obj_func(gdf, gdf1,run_dir):
    def calculate_ellipse(points):
        mean_center = np.mean(points, axis=0)
        med_center = np.median(points, axis=0)
        
        # Calculate covariance matrix
        cov = np.cov(points, rowvar=False)
        
        # Calculate eigenvalues and eigenvectors
        eigenvalues, eigenvectors = linalg.eigh(cov)
        
        # Sort by eigenvalues in decreasing order
        idx = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Calculate standard deviations along axes (for 95% confidence ellipse)
        std_dev = np.sqrt(eigenvalues) * 2
        
        # Calculate rotation angle
        rotation = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
        
        return mean_center, med_center, std_dev[0], std_dev[1], rotation

    # Assuming gdf and gdf1 are your GeoDataFrames
    # First, extract coordinates
    gdf['x'] = gdf['geometry'].x
    gdf['y'] = gdf['geometry'].y
    gdf1['x'] = gdf1['geometry'].x
    gdf1['y'] = gdf1['geometry'].y

    # Get points
    points = gdf[["x", "y"]].values
    points1 = gdf1[["x", "y"]].values

    # Calculate ellipse properties
    mean_center, med_center, major, minor, rotation = calculate_ellipse(points)
    mean_center1, med_center1, major1, minor1, rotation1 = calculate_ellipse(points1)

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 8))

    # Plot points from first GeoDataFrame - categorized by value
    gdf_value1 = gdf[gdf['value'] == 1]
    gdf_value2 = gdf[gdf['value'] == 2]
    ax.scatter(gdf_value1['x'], gdf_value1['y'], color='lightcoral', alpha=0.7, label='archaeology Value 1')
    ax.scatter(gdf_value2['x'], gdf_value2['y'], color='red', alpha=0.7, label='archaeology Value 2')

    # Plot points from second GeoDataFrame - categorized by value
    gdf1_value1 = gdf1[gdf1['value'] == 1]
    gdf1_value2 = gdf1[gdf1['value'] == 2]
    ax.scatter(gdf1_value1['x'], gdf1_value1['y'], color='lightblue', alpha=0.7, label='simulated Value 1')
    ax.scatter(gdf1_value2['x'], gdf1_value2['y'], color='blue', alpha=0.7, label='simulated Value 2')

    # Create and add ellipses to the plot
    ellipse = Ellipse(
        xy=mean_center, 
        width=major * 2, 
        height=minor * 2,
        angle=np.rad2deg(rotation), 
        facecolor="none",
        edgecolor="red",
        linestyle="--",
        linewidth=2,
        label="Std. Ellipse archaeology"
    )
    ax.add_patch(ellipse)

    ellipse_1 = Ellipse(
        xy=mean_center1, 
        width=major1 * 2, 
        height=minor1 * 2,
        angle=np.rad2deg(rotation1), 
        facecolor="none",
        edgecolor="blue",
        linestyle="--",
        linewidth=2,
        label="Std. Ellipse simulated"
    )
    ax.add_patch(ellipse_1)

    # Add mean centers to the plot
    ax.scatter(mean_center[0], mean_center[1], color='darkred', marker='*', s=100, label='archaeology Mean Center')
    ax.scatter(mean_center1[0], mean_center1[1], color='darkblue', marker='*', s=100, label='simulated Mean Center')

    # Set equal aspect so the ellipses look right
    ax.set_aspect('equal')

    # Add title and legend
    ax.set_title('Spatial Distribution Comparison with Standard Deviational Ellipses')
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

    # Calculate some metrics for the plot title or annotation
    vertices = ellipse.get_verts()
    ellipse_gdf = Polygon(vertices)
    vertices1 = ellipse_1.get_verts()
    ellipse_gdf1 = Polygon(vertices1)
    overlapping_area = ellipse_gdf.intersection(ellipse_gdf1).area
    total_non_overlapping_area = ellipse_gdf.symmetric_difference(ellipse_gdf1).area

    # Calculate overlap metric
    overlap_metric = (abs(overlapping_area - total_non_overlapping_area)) / (overlapping_area + total_non_overlapping_area) if (overlapping_area + total_non_overlapping_area) != 0 else 0
    overlap_score = 1 - overlap_metric

    # Add annotation with overlap information
    ax.annotate(f'Overlap score: {overlap_score:.4f}', 
                xy=(0.5, 0.02), 
                xycoords='axes fraction', 
                ha='center', 
                fontsize=12,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

    # Adjust layout and display
    plt.tight_layout()
    plt.show()

    # Print additional information
    #print(f"archaeology Ellipse - Major axis: {major:.4f}, Minor axis: {minor:.4f}, Rotation: {np.rad2deg(rotation):.2f}°")
    #print(f"simulated Ellipse - Major axis: {major1:.4f}, Minor axis: {minor1:.4f}, Rotation: {np.rad2deg(rotation1):.2f}°")
    #print(f"Overlap area: {overlapping_area:.4f}")
    #print(f"Non-overlapping area: {total_non_overlapping_area:.4f}")
    #print(f"Spatial overlap score: {overlap_score:.4f}")

    # Calculate value ratio metrics
    t1 = gdf[gdf['value'] == 1].shape[0]
    t2 = gdf[gdf['value'] == 2].shape[0]
    t_ratio = t2 / t1 if t1 != 0 else 0

    t11 = gdf1[gdf1['value'] == 1].shape[0]
    t21 = gdf1[gdf1['value'] == 2].shape[0]
    t1_ratio = t21 / t11 if t11 != 0 else 0

    # Calculate the absolute difference between ratios
    diff = abs(t_ratio - t1_ratio)
    # Normalize to 0-1 scale where 1 means identical ratios
    ratio_enc = 1 - diff


    #print(f"archaeology Value ratio (2:1): {t_ratio:.4f}")
    #print(f"simulated Value ratio (2:1): {t1_ratio:.4f}")
    #print(f"Value distribution similarity: {ratio_enc:.4f}")

    # Calculate total score
    total = (overlap_score + ratio_enc) / 2
    #print(f"Total similarity score: {total:.4f}")
    fig.savefig(os.path.join(run_dir, "spatial_similarity.png"))
    return total,ratio_enc

In [ ]:
total,ratio_enc=obj_func(gdf_real,gdf_model)

In [ ]:
def run_model_opt(model_years, wts, main_run_directory,y_output=y_output):
    """
    Runs the simulation for a specified number of years.

    Args:
        model_years: Number of years to simulate.
        wts: List of weights for environmental factors.
        y_output: Yearly output data for environmental factors.

    Returns:
        model: The completed model instance.
    """
    # Turn off interactive plotting globally at the beginning
    plt.ioff()

    # Generate a timestamp for the run directory
    current_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    # Create a list of indices and shuffle them for random yearly data selection
    inds = list(range(len(y_output)))
    random.shuffle(inds)

    # Generate suitability, return, and stress rasters for the initial year
    suitability_raster, return_raster, stress_ras = get_suitability_raster(y_output, inds, year=0, yearly_outputs=[], weights=wts)

    # Initialize the model with the generated rasters
    model = NomadModel(suitability_raster, return_raster, stress_ras, inds)

    # Create a directory for storing run results
    run_directory =os.path.join(main_run_directory,"run_{}".format(current_date))
    if not os.path.exists(run_directory):
        os.makedirs(run_directory)
    plot_y=[10,25,50]
    # Run the simulation for the specified number of years with progress bar
    for i in range(model_years):
        model.step()  # Advance the model by one year
        if i != (model_years - 1):
            if i in plot_y:
                viz_map(model, model.suitability_raster, i, run_directory)
            model.move_year(inds)
        else:
            viz_map(model, model.suitability_raster, i, run_directory)

    # Retrieve collected data from the model
    household_data = model.datacollector.get_agent_vars_dataframe()

    # Save data to CSV files
    household_data.to_csv(os.path.join(run_directory, "household_data.csv"))

    # Save weights to JSON files
    with open(os.path.join(run_directory, "permanent_weights_dict.json"), "w") as f:
        json.dump(permanent_weights_dict, f)
    with open(os.path.join(run_directory, "yearly_weights_dict.json"), "w") as f:
        json.dump(yearly_weights_dict, f)

    # Turn interactive mode back on at the end if needed for other visualizations
    #plt.ion()

    return model,run_directory

In [ ]:
# Define your objective function here (replace with your actual function)
def objective_function(permanent_weights_dict, yearly_weights_dict,trial_n, n_iterations=10,n_years=75, random_seed=None):
    """
    Your stochastic model objective function that returns a score from 0 to 1, where 1 is best.
    This function should use the weights provided in the dictionaries to compute the score.

    Args:
        permanent_weights_dict: Dictionary of permanent weights
        yearly_weights_dict: Dictionary of yearly weights
        n_iterations: Number of times to run the stochastic model
        random_seed: Optional random seed for reproducibility

    Returns:
        float: Average score across multiple runs (value between 0 and 1)
    """

    # Replace this with your actual implementation
    scores = []
    main_run_directory=r"C:\Users\Owner\Documents\Itay\thesis\ABM\opt\trial_{}".format(trial_n)
    # Set random seed if provided
    if random_seed is not None:
        np.random.seed(random_seed)

    # Run the model multiple times to account for stochasticity
    for i in range(n_iterations):
        weights = {**permanent_weights_dict, **yearly_weights_dict}
        wts=list(weights.values())
        sim,run_dir=run_model_opt(n_years, wts, main_run_directory)
        #define coords and co
        pext=Raster(param0)
        lowerLeft = arcpy.Point(pext.extent.XMin,pext.extent.YMin)
        lower_left_x=lowerLeft.X
        lower_left_y=lowerLeft.Y
        # Cell size (in meters)
        cell_size = 250
        gdf_model=to_gdf(lower_left_x,lower_left_y,cell_size,sim)
        gdf_real=gpd.read_file(r'C:\Users\Owner\Documents\Itay\thesis\ABM\points_all\all_P.shp')
        score,ratio_enc=obj_func(gdf_real,gdf_model,run_dir)
        score=1-score
        scores.append(score)

    # Return the average score across all iterations
    return np.mean(scores)  # Returns value between 0 and 1

def objective(trial):
    """
    Optuna objective function that samples parameter values and evaluates the stochastic model.
    """
    # Sample parameters for permanent weights
    sum_w=0
    #p_agri = trial.suggest_int("p_agri", 0,7)
    dist_to_kb = trial.suggest_int("dist_to_kb", 0, 7)
    sum_w+=dist_to_kb
    #aspect = trial.suggest_int("aspect", 0,7)
    p_water = trial.suggest_int("p_water", 0,7)
    sum_w+=p_water
    #p_veg_fit = trial.suggest_int("p_veg_fit", 0,7)
    mean_rain = trial.suggest_int("Mean_rain", 0,7)
    sum_w+=mean_rain
    #slope_range = trial.suggest_int("slope_range", 0,7)
    slope_suitability = trial.suggest_int("slope_suitability", 0,7)
    sum_w+=slope_suitability
    # Sample parameters for yearly weights
    return_to_site = trial.suggest_int("return_to_site", 0,7)
    sum_w+=return_to_site
    humen_stress = trial.suggest_int("humen_stress", 0,7)
    sum_w+=humen_stress
    #pasture_y = trial.suggest_int("pasture_y", 0,7)
    #yearly_agri = trial.suggest_int("yearly_agri", 0,7)
    #yearly_water = trial.suggest_int("yearly_water", 0,7)
    yearly_rain = trial.suggest_int("Yearly_rain", 0,7)
    sum_w+=yearly_rain
    #normalize to 0-1
    dist_to_kb/=sum_w
    p_water/=sum_w
    mean_rain/=sum_w
    slope_suitability/=sum_w
    return_to_site/=sum_w
    humen_stress/=sum_w
    yearly_rain/=sum_w
    # Create dictionaries to store weights for permanent and yearly factors
    permanent_weights_dict = {
        #"p_agri": wp1,  # Agricultural suitability
        "dist_to_kb": dist_to_kb,  # Distance to Kadesh Barnea
        #"aspect": wp3,  # Terrain aspect
        "p_water": p_water,  # Permanent water sources
        #"p_veg_fit": wp5,  # Vegetation suitability
        "Mean_rain": mean_rain,  # Mean annual rainfall
        #"slope_range": wp7,  # Slope range
        "slope_suitability": slope_suitability,  # Slope suitability
    }

    yearly_weights_dict = {
        "return_to_site": return_to_site,  # Return to previous sites
        "humen_stress": humen_stress,  # Human stress
        #"pasture_y": ws3,  # Pasture yield
        #"yearly_agri": ws4,  # Yearly agricultural suitability
        #"yearly_water": ws5,  # Yearly water availability
        "Yearly_rain": yearly_rain,  # Yearly rainfall
    }


    # Evaluate the objective function with these parameters
    # Run multiple iterations to account for stochasticity
    # Fixed random seed for each trial ensures reproducibility within a trial
    # but allows for different results between trials
    score = objective_function(
        permanent_weights_dict,
        yearly_weights_dict,
        trial.number,
        n_iterations=10,  # Run 10 iterations for each parameter set
        random_seed=trial.number  # Use trial number as seed for reproducibility
    )

    return score  # Optuna maximizes by default


def run_optimization(n_trials=100, n_jobs=1, show_progress_bar=True):
    """
    Run the Optuna optimization process for a stochastic model.

    Args:
        n_trials: Number of optimization trials to run
        n_jobs: Number of parallel jobs to run (-1 uses all cores)
        show_progress_bar: Whether to show a progress bar

    Returns:
        The best parameters and the best score
    """
    # Create a pruner to terminate unpromising trials early
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)

    # Create a study object and optimize the objective function
    study = optuna.create_study(
        direction="minimize",
        pruner=pruner,
        sampler=optuna.samplers.TPESampler(seed=42)  # Use TPE sampler with fixed seed for reproducibility
    )

    # Run optimization
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=show_progress_bar
    )

    # Get the best parameters
    best_params = study.best_params
    best_score = study.best_value

    # Extract the best weights into the dictionaries format
    best_permanent_weights = {
        #"p_agri": best_params["p_agri"],
        "dist_to_kb": best_params["dist_to_kb"],
        #"aspect": best_params["aspect"],
        "p_water": best_params["p_water"],
        #"p_veg_fit": best_params["p_veg_fit"],
        "Mean_rain": best_params["Mean_rain"],
        #"slope_range": best_params["slope_range"],
        "slope_suitability": best_params["slope_suitability"],
    }

    best_yearly_weights = {
        "return_to_site": best_params["return_to_site"],
        "humen_stress": best_params["humen_stress"],
        #"pasture_y": best_params["pasture_y"],
        #"yearly_agri": best_params["yearly_agri"],
        #"yearly_water": best_params["yearly_water"],
        "Yearly_rain": best_params["Yearly_rain"],
    }

    return best_permanent_weights, best_yearly_weights, best_score



In [ ]:
def objective1(trial):
    """
    Optuna objective function that samples parameter values and evaluates the stochastic model.
    """
    # Sample parameters for permanent weights
    sum_w=0
    #p_agri = trial.suggest_int("p_agri", 0,7)
    dist_to_kb = trial.suggest_int("dist_to_kb", 0, 7)
    sum_w+=dist_to_kb
    #aspect = trial.suggest_int("aspect", 0,7)
    p_water = trial.suggest_int("p_water", 0,7)
    sum_w+=p_water
    #p_veg_fit = trial.suggest_int("p_veg_fit", 0,7)
    mean_rain = trial.suggest_int("Mean_rain", 0,7)
    sum_w+=mean_rain
    #slope_range = trial.suggest_int("slope_range", 0,7)
    slope_suitability = trial.suggest_int("slope_suitability", 0,7)
    sum_w+=slope_suitability
    # Sample parameters for yearly weights
    return_to_site = trial.suggest_int("return_to_site", 0,7)
    sum_w+=return_to_site
    humen_stress = trial.suggest_int("humen_stress", 0,7)
    sum_w+=humen_stress
    #pasture_y = trial.suggest_int("pasture_y", 0,7)
    #yearly_agri = trial.suggest_int("yearly_agri", 0,7)
    #yearly_water = trial.suggest_int("yearly_water", 0,7)
    yearly_rain = trial.suggest_int("Yearly_rain", 0,7)
    sum_w+=yearly_rain
    #normalize to 0-1
    dist_to_kb/=sum_w
    p_water/=sum_w
    mean_rain/=sum_w
    slope_suitability/=sum_w
    return_to_site/=sum_w
    humen_stress/=sum_w
    yearly_rain/=sum_w
    # Create dictionaries to store weights for permanent and yearly factors
    permanent_weights_dict = {
        #"p_agri": wp1,  # Agricultural suitability
        "dist_to_kb": dist_to_kb,  # Distance to Kadesh Barnea
        #"aspect": wp3,  # Terrain aspect
        "p_water": p_water,  # Permanent water sources
        #"p_veg_fit": wp5,  # Vegetation suitability
        "Mean_rain": mean_rain,  # Mean annual rainfall
        #"slope_range": wp7,  # Slope range
        "slope_suitability": slope_suitability,  # Slope suitability
    }

    yearly_weights_dict = {
        "return_to_site": return_to_site,  # Return to previous sites
        "humen_stress": humen_stress,  # Human stress
        #"pasture_y": ws3,  # Pasture yield
        #"yearly_agri": ws4,  # Yearly agricultural suitability
        #"yearly_water": ws5,  # Yearly water availability
        "Yearly_rain": yearly_rain,  # Yearly rainfall
    }


    # Evaluate the objective function with these parameters
    # Run multiple iterations to account for stochasticity
    # Fixed random seed for each trial ensures reproducibility within a trial
    # but allows for different results between trials
    score = objective_function(
        permanent_weights_dict,
        yearly_weights_dict,
        trial.number,
        n_iterations=2,  # Run 10 iterations for each parameter set
        n_years=7,
        random_seed=trial.number  # Use trial number as seed for reproducibility
    )

    return score  # Optuna maximizes by default


def run_optimization1(n_trials=3, n_jobs=1, show_progress_bar=True):
    """
    Run the Optuna optimization process for a stochastic model.

    Args:
        n_trials: Number of optimization trials to run
        n_jobs: Number of parallel jobs to run (-1 uses all cores)
        show_progress_bar: Whether to show a progress bar

    Returns:
        The best parameters and the best score
    """
    # Create a pruner to terminate unpromising trials early
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)

    # Create a study object and optimize the objective function
    study = optuna.create_study(
        direction="minimize",
        pruner=pruner,
        sampler=optuna.samplers.TPESampler(seed=42)  # Use TPE sampler with fixed seed for reproducibility
    )

    # Run optimization
    study.optimize(
        objective1,
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=show_progress_bar
    )

    # Get the best parameters
    best_params = study.best_params
    best_score = study.best_value

    # Extract the best weights into the dictionaries format
    best_permanent_weights = {
        #"p_agri": best_params["p_agri"],
        "dist_to_kb": best_params["dist_to_kb"],
        #"aspect": best_params["aspect"],
        "p_water": best_params["p_water"],
        #"p_veg_fit": best_params["p_veg_fit"],
        "Mean_rain": best_params["Mean_rain"],
        #"slope_range": best_params["slope_range"],
        "slope_suitability": best_params["slope_suitability"],
    }

    best_yearly_weights = {
        "return_to_site": best_params["return_to_site"],
        "humen_stress": best_params["humen_stress"],
        #"pasture_y": best_params["pasture_y"],
        #"yearly_agri": best_params["yearly_agri"],
        #"yearly_water": best_params["yearly_water"],
        "Yearly_rain": best_params["Yearly_rain"],
    }

    return best_permanent_weights, best_yearly_weights, best_score

In [ ]:
try_opt=run_optimization1()

In [ ]:
try_opt

In [ ]:

if __name__ == "__main__":
    # Run the optimization
    best_permanent_weights, best_yearly_weights, best_score = run_optimization(
        n_trials=100,
        n_jobs=-1,  # Use all available cores
        show_progress_bar=True
    )
    
    print(f"Best Score: {best_score:.4f}")
    print("\nBest Permanent Weights:")
    for key, value in best_permanent_weights.items():
        print(f"  {key}: {value:.4f}")
    
    print("\nBest Yearly Weights:")
    for key, value in best_yearly_weights.items():
        print(f"  {key}: {value:.4f}")
    
    # Verify the robustness of the solution
    print("\nVerifying robustness of the solution...")
    verification_score = objective_function(
        best_permanent_weights, 
        best_yearly_weights,
        n_iterations=30,  # Use more iterations for final verification
        random_seed=None  # No fixed seed for true randomness
    )
    print(f"Verification Score (30 iterations): {verification_score:.4f}")
    
    # You can also visualize the optimization process
    print("\nTo visualize the optimization process, you can use:")
    print("optuna.visualization.plot_optimization_history(study)")
    print("optuna.visualization.plot_param_importances(study)")

# Optional: More comprehensive visualization and analysis
# Uncomment if you want to include visualization
"""
import matplotlib.pyplot as plt
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_contour,
    plot_slice
)

# Save the study object from optimization
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5),
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=100, n_jobs=-1, show_progress_bar=True)

# Plot the optimization history
fig1 = plot_optimization_history(study)
fig1.show()

# Plot parameter importances
fig2 = plot_param_importances(study)
fig2.show()

# Plot parallel coordinate plot
fig3 = plot_parallel_coordinate(study)
fig3.show()

# Plot contour plot for some important parameters
# (replace with the most important parameters from the importance plot)
fig4 = plot_contour(study, params=["p_agri", "dist_to_kb"])
fig4.show()

# Plot slice plot
fig5 = plot_slice(study)
fig5.show()

# Additional analysis - parameter correlations
param_values = {}
for param_name in study.best_params.keys():
    param_values[param_name] = []

for trial in study.trials:
    for param_name, param_value in trial.params.items():
        param_values[param_name].append(param_value)

# Calculate correlation matrix
import pandas as pd
import seaborn as sns

param_df = pd.DataFrame(param_values)
correlation_matrix = param_df.corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Parameter Correlation Matrix')
plt.tight_layout()
plt.show()

# Testing best parameters with different random seeds
print("Testing robustness across different random seeds...")
test_scores = []
for seed in range(10):
    score = objective_function(
        {k: study.best_params[k] for k in best_permanent_weights.keys()},
        {k: study.best_params[k] for k in best_yearly_weights.keys()},
        n_iterations=10,
        random_seed=seed
    )
    test_scores.append(score)

print(f"Average score: {np.mean(test_scores):.4f}")
print(f"Standard deviation: {np.std(test_scores):.4f}")
print(f"Min score: {min(test_scores):.4f}")
print(f"Max score: {max(test_scores):.4f}")
"""